In [7]:
# ======================= Beam + ACC (+CKG prior) Evaluator =======================
# Evaluates ONLY valid/test splits; no per-graph chain dumps.
# Saves everything under: out/beam_acc_eval_<timestamp>/

import os, json, math, time, sys
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

# ------------------- CONFIG (edit paths as needed) -------------------
CKPT_PATH   = r"out/cvul/1760989836/model.pt"       # your trained model
VALID_DIR   = r"Dataset/valid/hetero_ready_gcbert"  # evaluation split: valid
TEST_DIR    = r"Dataset/test/hetero_ready_gcbert"   # evaluation split: test
CKG_PATH    = r"ckg/ckg.json"                       # optional; set to None to ignore CKG
EVAL_LIMIT  = None                                  # None=all, or set an int cap per split

# Beam & prior (aligned with your training)
SEED_K        = 8
BEAM_WIDTH    = 24
BEAM_MAX_HOPS = 5
ALPHA_NODE    = 0.7
LAMBDA_PRIOR  = 0.15          # CKG weight; set 0 to disable effect while still reporting

# ACC options
ENFORCE_ACC         = True    # set False to relax ACC (max throughput)
NO_CYCLES           = True
REPEAT_REL_PENALTY  = 0.05    # small penalty per immediate relation repeat (helps diversity)

# ------------------- Device / speed knobs -------------------
def pick_device(min_free_mb=256):
    if not torch.cuda.is_available(): return "cpu"
    try:
        free,_ = torch.cuda.mem_get_info()
        return "cuda" if (free//(1024**2)) >= min_free_mb else "cpu"
    except:
        return "cuda"

DEVICE = pick_device()
torch.backends.cuda.matmul.allow_tf32 = True
print(f"[env] torch={torch.__version__} device={DEVICE}")

# ------------------- IO helpers -------------------
def safe_load(p: Path):
    if p.suffix.lower()==".json":
        return json.loads(p.read_text(encoding="utf-8"))
    return torch.load(str(p), map_location="cpu")

def list_files(root_dir, patterns=("*.json","*.pt"), json_first=True):
    root = Path(root_dir)
    files = []
    for pat in (["*.json","*.pt"] if json_first else patterns):
        files.extend(sorted(root.glob(pat)))
    return files

# ------------------- Normalization -------------------
def _first_2d_float(arr):
    t = torch.as_tensor(arr).float()
    if t.ndim == 1: t = t.view(-1, 1)
    return t

def to_long_2(e):
    if e is None: return torch.zeros((2,0), dtype=torch.long)
    t=torch.as_tensor(e)
    if t.ndim==2 and t.shape[0]==2: return t.long().contiguous()
    if t.ndim==2 and t.shape[1]==2: return t.t().long().contiguous()
    if isinstance(e,(list,tuple)) and len(e)==2:
        s=torch.as_tensor(e[0]).view(-1).long()
        d=torch.as_tensor(e[1]).view(-1).long()
        return torch.stack([s,d], dim=0)
    if t.numel()==0: return torch.zeros((2,0), dtype=torch.long)
    raise RuntimeError("edge_index must be [2,E], [E,2], or (src,dst)")

def sanitize_edges(N:int, ei:torch.Tensor):
    if ei is None or ei.numel()==0:
        return torch.zeros((2,0), dtype=torch.long)
    s,d = ei
    m=(s>=0)&(s<N)&(d>=0)&(d<N)
    if m.any():
        return torch.stack([s[m], d[m]], dim=0)
    return torch.zeros((2,0), dtype=torch.long)

# accepted relations
BASE_REL = ["DFG","CFG","CALL","ARG2PARAM","RET2CALL","RET2LHS"]

def _get_store_feat(store):
    # Prefer GraphCodeBERT fields if present
    for nm in ["gcbert","gcb_x","x","x_text","x_num","features","feat","emb"]:
        if hasattr(store, nm):
            val = getattr(store, nm)
            if val is not None:
                t = _first_2d_float(val)
                if t.numel() > 0:
                    return t
    if hasattr(store, "__dict__"):
        for nm in ["gcbert","gcb_x","x","x_text","x_num","features","feat","emb"]:
            if nm in store.__dict__ and store.__dict__[nm] is not None:
                t = _first_2d_float(store.__dict__[nm])
                if t.numel() > 0:
                    return t
    return None

def normalize_hetero(obj, RELATIONS, ADD_SUMMARY_EDGES=True):
    node_stores = getattr(obj, "node_stores", None)
    edge_stores = getattr(obj, "edge_stores", None)
    if node_stores is None or edge_stores is None:
        return None

    candidates=[]
    for st in node_stores:
        feat = _get_store_feat(st)
        if feat is not None and feat.numel() > 0:
            key = getattr(st, "_key", None) or getattr(st, "type", None) or "code"
            candidates.append((key, feat))
    if not candidates:
        return None

    nt, x = max(candidates, key=lambda kv: kv[1].size(0))
    N = x.size(0)

    E = {r: torch.zeros((2,0), dtype=torch.long) for r in RELATIONS}
    for es in edge_stores:
        ei = getattr(es, "edge_index", None)
        if ei is None: continue
        src_t = getattr(es, "src_type", None)
        dst_t = getattr(es, "dst_type", None)
        if (src_t is not None and dst_t is not None) and not (src_t == nt and dst_t == nt):
            continue
        rel = getattr(es, "edge_type", None) or getattr(es, "_key", None)
        if isinstance(rel, (tuple, list)) and len(rel) == 3:
            rel = str(rel[1])
        rel = str(rel).upper().split("__")[-1] if rel is not None else None
        if rel in E:
            E[rel] = sanitize_edges(N, to_long_2(ei))

    if ADD_SUMMARY_EDGES:
        base = E.get("DFG", torch.zeros((2,0), dtype=torch.long))
        for r in ("ARG2PARAM", "RET2CALL", "RET2LHS"):
            if r in E and E[r].numel() > 0:
                base = torch.cat([base, E[r]], dim=1)
        E["DFG_THIN"] = base

    # Labels are optional for eval; we'll compute node metrics if available
    y = None
    for st in node_stores:
        st_key = getattr(st, "_key", None) or getattr(st, "type", None)
        if st_key == nt and hasattr(st, "y") and st.y is not None:
            yy = torch.as_tensor(st.y).float().view(-1)
            if yy.numel() == N: y = yy
            break

    return {"x": x, "edges": E, "y": y, "source": "default"}

def coerce_x_any(obj):
    for k in ["gcbert","gcb_x","x","features","node_features","feat","emb","x_text","x_num","x_dense","x_numeric"]:
        if isinstance(obj, dict) and k in obj and obj[k] is not None:
            t=torch.as_tensor(obj[k]).float()
            if t.ndim==1: t=t.view(-1,1)
            return t
    if isinstance(obj, dict) and "nodes" in obj and isinstance(obj["nodes"], list) and obj["nodes"]:
        rows=[]
        for nd in obj["nodes"]:
            if not isinstance(nd, dict): continue
            for k in ["gcbert","gcb_x","x","feat","emb","features"]:
                if k in nd and nd[k] is not None:
                    rows.append(torch.as_tensor(nd[k]).float().view(1,-1)); break
        if rows:
            d=max(r.size(1) for r in rows)
            rows=[F.pad(r,(0,d-r.size(1))) for r in rows]
            return torch.cat(rows, dim=0)
    return None

def build_edges_any(obj, N, RELATIONS, ADD_SUMMARY_EDGES=True):
    E={r:torch.zeros((2,0),dtype=torch.long) for r in RELATIONS}
    if isinstance(obj, dict) and "edges" in obj and isinstance(obj["edges"], dict):
        for r in RELATIONS:
            if r in obj["edges"]:
                E[r]=sanitize_edges(N, to_long_2(obj["edges"][r]))
        if ADD_SUMMARY_EDGES:
            base=E.get("DFG", torch.zeros((2,0),dtype=torch.long))
            for r in ("ARG2PARAM","RET2CALL","RET2LHS"):
                if E[r].numel(): base=torch.cat([base,E[r]], dim=1)
            E["DFG_THIN"]=base
        return E
    for r in RELATIONS:
        for k in [r, f"{r}_edge_index", f"edge_index_{r}", f"{r.lower()}_edge_index", f"edge_index_{r.lower()}"]:
            if isinstance(obj, dict) and k in obj:
                E[r]=sanitize_edges(N, to_long_2(obj[k])); break
    if ADD_SUMMARY_EDGES:
        base=E.get("DFG", torch.zeros((2,0),dtype=torch.long))
        for r in ("ARG2PARAM","RET2CALL","RET2LHS"):
            if E[r].numel(): base=torch.cat([base,E[r]], dim=1)
        E["DFG_THIN"]=base
    return E

def normalize_graph(obj, RELATIONS, ADD_SUMMARY_EDGES=True):
    if isinstance(obj, dict) and "graph" in obj and isinstance(obj["graph"], dict):
        obj = obj["graph"]
    if "torch_geometric" in str(type(obj)) and not isinstance(obj, dict):
        g_het = normalize_hetero(obj, RELATIONS, ADD_SUMMARY_EDGES)
        if g_het is not None:
            return g_het
        if hasattr(obj, "x") and obj.x is not None:
            x = _first_2d_float(obj.x); N=x.size(0)
            E={r:torch.zeros((2,0),dtype=torch.long) for r in RELATIONS}
            if hasattr(obj,"edge_index"):
                E["DFG"]=sanitize_edges(N, to_long_2(obj.edge_index))
            if ADD_SUMMARY_EDGES:
                base=E.get("DFG", torch.zeros((2,0),dtype=torch.long))
                for r in ("ARG2PARAM","RET2CALL","RET2LHS"):
                    E[r]=E.get(r, torch.zeros((2,0),dtype=torch.long))
                E["DFG_THIN"]=base
            return {"x":x,"edges":E,"y":None,"source":"default"}
        return None
    if isinstance(obj, dict):
        x = coerce_x_any(obj)
        if x is None or x.numel()==0:
            return None
        N = x.size(0)
        E = build_edges_any(obj, N, RELATIONS, ADD_SUMMARY_EDGES)
        y=None
        for k in ["y","label","labels","vulnerable","is_sink","target","targets"]:
            if k in obj and obj[k] is not None:
                try:
                    yy=torch.as_tensor(obj[k]).float().view(-1)
                    if yy.numel()==N: y=yy
                except: pass
                break
        return {"x":x, "edges":E, "y":y, "source":"default"}
    return None

# ------------------- Model -------------------
class GraphBlock(nn.Module):
    def __init__(self, hidden, relations):
        super().__init__()
        self.relations=relations
        self.lin_rel=nn.ModuleDict({r:nn.Linear(hidden,hidden,bias=False) for r in relations})
        self.lin_self=nn.Linear(hidden,hidden)
    def forward(self, h, E):
        H=h
        for r in self.relations:
            ei=E.get(r)
            if ei is None or ei.numel()==0: continue
            s,d=ei
            msg=self.lin_rel[r](H)
            agg=torch.zeros_like(H)
            agg.index_add_(0, d, msg[s])
            H=H+agg
        return self.lin_self(H)

class CausalVulNet(nn.Module):
    def __init__(self, hidden=64, layers=3, relations=None):
        super().__init__()
        if relations is None:
            relations = BASE_REL + ["DFG_THIN"]
        self.relations = list(relations)
        self.proj_cache = nn.ModuleDict()
        self.blocks = nn.ModuleList([GraphBlock(hidden, self.relations) for _ in range(layers)])
        self.node_head = nn.Linear(hidden,1)
        self.seed_head = nn.Linear(hidden,1)
        self.rel_gate  = nn.ParameterDict({r: nn.Parameter(torch.tensor(0.0)) for r in self.relations})
        self.edge_bilin= nn.Parameter(torch.empty(hidden, hidden)); nn.init.xavier_uniform_(self.edge_bilin)
        self.hidden = hidden; self.layers = layers

    def _proj(self, D:int):
        k=str(D)
        if k not in self.proj_cache:
            layer=nn.Linear(D, self.hidden).to(next(self.parameters()).device)
            self.proj_cache[k]=layer
        return self.proj_cache[k]

    def encode(self, x, E):
        h=F.relu(self._proj(x.size(1))(x))
        for blk in self.blocks: h=F.elu(blk(h,E))
        return h

    def edge_scores(self, h, E):
        out={}
        for r,ei in E.items():
            if ei is None or ei.numel()==0:
                out[r]=torch.zeros((0,), device=h.device); continue
            s,d=ei
            hs=h[s] @ self.edge_bilin
            out[r]=(hs*h[d]).sum(dim=1) + self.rel_gate[r]
        return out

    def forward_full(self, x, E):
        seed_h=F.relu(self._proj(x.size(1))(x))
        seed_logit=self.seed_head(seed_h).squeeze(-1)
        h=self.encode(x,E)
        node_logit=self.node_head(h).squeeze(-1)
        edge_sc=self.edge_scores(h,E)
        return seed_logit, node_logit, h, edge_sc

def load_model(ckpt_path:str):
    ckpt = torch.load(ckpt_path, map_location="cpu")
    hidden    = ckpt.get("hidden", 64)
    layers    = ckpt.get("layers", 3)
    relations = ckpt.get("relations", BASE_REL + ["DFG_THIN"])
    model = CausalVulNet(hidden=hidden, layers=layers, relations=relations).to(DEVICE)
    model.load_state_dict(ckpt["state_dict"], strict=False)
    model.eval()
    base_rel = [r for r in relations if r != "DFG_THIN"]
    add_summary = ("DFG_THIN" in relations)
    return model, base_rel, add_summary

# ------------------- Beam + ACC (+CKG) -------------------
@dataclass
class BeamPath:
    score: float
    nodes: List[int]
    rels:  List[str]

def build_adj(E):
    adj_out={r:{} for r in E}; adj_in={r:{} for r in E}
    for r,ei in E.items():
        if ei is None or ei.numel()==0: continue
        s,d=ei; ss,dd=s.tolist(), d.tolist()
        for u,v in zip(ss,dd):
            adj_out[r].setdefault(u,[]).append(v)
            adj_in [r].setdefault(v,[]).append(u)
    return adj_out, adj_in

def pick_seeds(seed_logit, E, k):
    N=seed_logit.numel()
    deg=torch.zeros(N, device=seed_logit.device)
    for ei in E.values():
        if ei is None or ei.numel()==0: continue
        s,_=ei; deg.index_add_(0, s, torch.ones_like(s, dtype=deg.dtype))
    cand=torch.where(deg>0)[0]
    if cand.numel()==0: return torch.topk(seed_logit, k=min(k,N)).indices.tolist()
    k=min(k, cand.numel()); vals=seed_logit[cand]
    return cand[torch.topk(vals,k=k).indices].tolist()

def _edge_uv_scores(edge_sc, E, N:int):
    uv={}
    for r,ei in E.items():
        if ei is None or ei.numel()==0: uv[r]={}; continue
        s,d=ei; es=edge_sc[r].detach().float()
        mp={}
        for i in range(s.numel()):
            u=int(s[i]); v=int(d[i])
            if 0<=u<N and 0<=v<N:
                val=float(es[i].item())
                mp[(u,v)] = max(mp.get((u,v), val), val)
        uv[r]=mp
    return uv

# ACC checks (pragmatic)
def acc_ok(prev_rel: Optional[str], rel: str, call_depth: int) -> Tuple[bool,int]:
    # Simple inter-procedural discipline:
    # CALL increases depth; RET2CALL/RET2LHS decrease if depth>0; ARG2PARAM ok around calls.
    d = call_depth
    if rel == "CALL":
        d += 1; return True, d
    if rel in ("RET2CALL","RET2LHS"):
        return (d > 0), (max(0, d-1))
    # Intra-proc hops
    return True, d

def run_beam_ckg_acc(p, edge_sc, E, seeds, ckg=None, width=24, max_hops=5,
                     alpha_node=0.7, lambda_prior=0.15, enforce_acc=True, no_cycles=True):
    N=p.numel()
    adj_out, adj_in = build_adj(E)
    uv = _edge_uv_scores(edge_sc, E, N)

    # CKG priors (log space)
    ep = (ckg or {}).get("edge_prior_prob", {})
    bp = (ckg or {}).get("bigram_prob", {})
    motifs = set()
    for m in (ckg or {}).get("motifs_topk", []):
        seq = tuple(m.get("rels",[]))
        if len(seq)==3: motifs.add(seq)

    def logp_edge(r):       return math.log(max(float(ep.get(r, 1e-6)), 1e-9))
    def logp_bigram(a, b):  return math.log(max(float(bp.get(a, {}).get(b, 1e-6)), 1e-9))
    def clog(x):            return float(torch.log(x.clamp(1e-9,1-1e-9)))

    beams=[(BeamPath(clog(p[s]), [int(s)], []), 0) for s in seeds if 0<=int(s)<N]  # (path, call_depth)
    if not beams: return []

    out=[]
    for _ in range(max_hops):
        nxt=[]
        for (b, depth) in beams:
            u=b.nodes[-1]
            # enumerate both out/in edges around u (undirected hop budget like training)
            cand=[]
            for r in E.keys():
                for v in adj_out[r].get(u, []): cand.append((r,u,v))
                for v in adj_in [r].get(u, []): cand.append((r,v,u))
            if not cand:
                out.append((b, depth)); continue
            prev_r = b.rels[-1] if b.rels else None
            for (r,uu,vv) in cand:
                if not (0<=vv<N): continue
                if no_cycles and vv in b.nodes: 
                    continue
                # ACC check
                if enforce_acc:
                    ok, new_depth = acc_ok(prev_r, r, depth)
                    if not ok: 
                        continue
                else:
                    new_depth = depth

                es = uv.get(r,{}).get((uu,vv), 0.0)
                base = b.score + alpha_node*clog(p[vv]) + (1-alpha_node)*es

                # small penalty for immediate relation repetition
                if b.rels and r == b.rels[-1]:
                    base -= REPEAT_REL_PENALTY

                # CKG priors
                if ckg is not None and lambda_prior>0:
                    base += lambda_prior * logp_edge(r)
                    if prev_r is not None:
                        base += lambda_prior * logp_bigram(prev_r, r)
                    # motif bonus when we complete a tri-gram
                    if len(b.rels) >= 2:
                        tri = (b.rels[-2], b.rels[-1], r)
                        if tri in motifs:
                            base += 0.10  # tiny extra bonus for top motifs

                nxt.append((BeamPath(base, b.nodes+[vv], b.rels+[r]), new_depth))
        if not nxt: break
        nxt.sort(key=lambda x:x[0].score, reverse=True)
        beams = nxt[:width]
    out.extend(beams); out.sort(key=lambda x:x[0].score, reverse=True)
    return [bp for (bp,_) in out[:width]]

# Fallback to plain beam if ACC filtered everything
def run_beam_plain(p, edge_sc, E, seeds, width=24, max_hops=5, alpha_node=0.7):
    N=p.numel()
    adj_out, adj_in = build_adj(E)
    uv = _edge_uv_scores(edge_sc, E, N)
    def clog(x): return float(torch.log(x.clamp(1e-9,1-1e-9)))
    beams=[BeamPath(clog(p[s]), [int(s)], []) for s in seeds if 0<=int(s)<N]
    if not beams: return []
    out=[]
    for _ in range(max_hops):
        nxt=[]
        for b in beams:
            u=b.nodes[-1]
            cand=[]
            for r in E.keys():
                for v in adj_out[r].get(u, []): cand.append((r,u,v))
                for v in adj_in [r].get(u, []): cand.append((r,v,u))
            if not cand: out.append(b); continue
            for (r,uu,vv) in cand:
                if not (0<=vv<N): continue
                es = uv.get(r,{}).get((uu,vv), 0.0)
                sc = b.score + alpha_node*clog(p[vv]) + (1-alpha_node)*es
                nxt.append(BeamPath(sc, b.nodes+[vv], b.rels+[r]))
        if not nxt: break
        nxt.sort(key=lambda x:x.score, reverse=True)
        beams = nxt[:width]
    out.extend(beams); out.sort(key=lambda x:x.score, reverse=True)
    return out[:width]

# ------------------- Metrics -------------------
def f1_from_probs(p: torch.Tensor, y: torch.Tensor, thr: float):
    if y is None or y.numel()!=p.numel(): return None
    yb=(y>0.5); pb=(p>thr)
    tp=(pb&yb).sum().item(); fp=(pb&~yb).sum().item(); fn=(~pb&yb).sum().item()
    prec=tp/(tp+fp+1e-9); rec=tp/(tp+fn+1e-9); f1=2*prec*rec/(prec+rec+1e-9)
    return {"precision":prec,"recall":rec,"f1":f1}

@torch.no_grad()
def calibrate_thresholds(model, files, base_rel, add_summary, per_source_default=0.25, limit=None):
    # Simple: collect probs/labels from up to 'limit' graphs, pick best F1 over grid
    P=[]; Y=[]
    n=0
    pbar = tqdm(files[:(limit or len(files))], desc="[calibrate]", unit="graph")
    for fp in pbar:
        try:
            obj=safe_load(fp); g=normalize_graph(obj, base_rel, add_summary)
            if g is None or g["x"] is None or g["y"] is None: continue
            x = g["x"].to(DEVICE); E={r:e.to(DEVICE) for r,e in g["edges"].items()}
            _, nl, _, _ = model.forward_full(x, E)
            P.append(torch.sigmoid(nl).detach().cpu()); Y.append(g["y"].float().view(-1))
            n+=1
        except: 
            continue
    if not P: 
        return {"default": per_source_default}
    P=torch.cat(P); Y=torch.cat(Y)
    best_thr, best_f = 0.25, -1.0
    for thr in [i/100 for i in range(5,96,5)]:
        m=f1_from_probs(P, Y, thr)
        if m and m["f1"]>best_f:
            best_thr, best_f = thr, m["f1"]
    return {"default": best_thr}

def compute_CFAM(model, g_cpu, paths: List[BeamPath]):
    if not paths: return None
    x = g_cpu["x"].to(DEVICE).detach().requires_grad_(True)
    with torch.enable_grad():
        _, nl, _, _ = model.forward_full(x, {r:e.to(DEVICE) for r,e in g_cpu["edges"].items()})
        s = torch.sigmoid(nl).mean()
        s.backward()
        gn = x.grad.detach().abs().sum(dim=1)
    causal=set(n for bp in paths for n in bp.nodes if 0<=n<x.size(0))
    if not causal: return None
    mask=torch.zeros(x.size(0), dtype=torch.bool, device=gn.device)
    mask[torch.tensor(list(causal), device=gn.device)] = True
    num=gn[mask].sum().item(); den=gn.sum().item()+1e-9
    return num/den

def compute_CCS(model, g_cpu, paths: List[BeamPath]):
    if not paths: return None
    x = g_cpu["x"].to(DEVICE)
    with torch.no_grad():
        _, nl, _, _ = model.forward_full(x, {r:e.to(DEVICE) for r,e in g_cpu["edges"].items()})
        p0 = torch.sigmoid(nl).mean().item()
    cf = x.clone()
    causal = sorted(set(n for bp in paths for n in bp.nodes if 0<=n<x.size(0)))
    if causal:
        cf[torch.tensor(causal, device=cf.device)] = 0.0
    with torch.no_grad():
        _, nl2, _, _ = model.forward_full(cf, {r:e.to(DEVICE) for r,e in g_cpu["edges"].items()})
        p1 = torch.sigmoid(nl2).mean().item()
    return (p0 - p1) ** 2

# ------------------- Eval Loop -------------------
def to_device_graph(g):
    return {"x": g["x"].to(DEVICE),
            "edges": {r:e.to(DEVICE) for r,e in g["edges"].items()},
            "y": (g.get("y").to(DEVICE) if g.get("y") is not None else None),
            "source": g.get("source","default")}

def load_ckg(path_or_none):
    if not path_or_none: return None
    p=Path(path_or_none)
    if not p.exists(): 
        print(f"[warn] CKG not found at {p}, continuing without it."); return None
    return json.loads(p.read_text(encoding="utf-8"))

def eval_split(name, data_dir, model, base_rel, add_summary, ckg, thresholds):
    files = list_files(data_dir, json_first=True)
    if EVAL_LIMIT: files = files[:EVAL_LIMIT]
    if not files:
        return {"split":name, "n_graphs_total":0, "n_graphs_used":0,
                "avg_beam_length":0.0,"interproc_step_share":0.0,"motif_match_share":0.0,
                "CFAM_mean":None,"CCS_mean":None,"precision":None,"recall":None,"f1":None}

    motifs = set()
    if ckg:
        for m in ckg.get("motifs_topk", []):
            seq = tuple(m.get("rels",[]))
            if len(seq)==3: motifs.add(seq)

    used=0
    s_beam_len=0.0; s_inter=0; s_steps=0; s_motif=0
    cfam_vals=[]; ccs_vals=[]
    # for node metrics
    sum_prec=0.0; sum_rec=0.0; sum_f1=0.0; n_metric=0

    pbar = tqdm(files, desc=f"[eval:{name}]", unit="graph")
    for fp in pbar:
        try:
            obj = safe_load(fp); g_cpu = normalize_graph(obj, base_rel, add_summary)
            if g_cpu is None or g_cpu["x"] is None or g_cpu["x"].numel()==0:
                continue
            g = to_device_graph(g_cpu)
            with torch.no_grad():
                sd, nl, h, es = model.forward_full(g["x"], g["edges"])
                seeds = pick_seeds(sd.detach(), g["edges"], SEED_K)
                p  = torch.sigmoid(nl)
                beams = run_beam_ckg_acc(p, es, g["edges"], seeds, ckg=ckg, width=BEAM_WIDTH,
                                         max_hops=BEAM_MAX_HOPS, alpha_node=ALPHA_NODE,
                                         lambda_prior=LAMBDA_PRIOR, enforce_acc=ENFORCE_ACC, no_cycles=NO_CYCLES)
                if not beams:
                    # fallback to plain beam so we still use this graph
                    beams = run_beam_plain(p, es, g["edges"], seeds, width=BEAM_WIDTH,
                                           max_hops=BEAM_MAX_HOPS, alpha_node=ALPHA_NODE)
            if not beams:
                continue  # truly nothing to use

            # aggregate beam stats
            used += 1
            avg_len = sum(len(b.nodes) for b in beams)/(len(beams) or 1)
            s_beam_len += avg_len

            # inter-proc & motifs
            inter_edges={"CALL","ARG2PARAM","RET2CALL","RET2LHS"}
            steps=0; inter=0; motif_hits=0
            for b in beams:
                steps += max(0, len(b.rels))
                inter += sum(1 for r in b.rels if r in inter_edges)
                for i in range(2, len(b.rels)):
                    if (b.rels[i-2], b.rels[i-1], b.rels[i]) in motifs:
                        motif_hits += 1
            s_steps += steps
            s_inter += inter
            s_motif += motif_hits

            # CFAM / CCS
            cf = compute_CFAM(model, g_cpu, beams); cc = compute_CCS(model, g_cpu, beams)
            if cf is not None: cfam_vals.append(cf)
            if cc is not None: ccs_vals.append(cc)

            # classic metrics if labels exist
            if g_cpu["y"] is not None and g_cpu["y"].numel()==p.numel():
                thr = thresholds.get("default", 0.25)
                m = f1_from_probs(p.detach().cpu(), g_cpu["y"].cpu().float().view(-1), thr)
                if m:
                    sum_prec += m["precision"]; sum_rec += m["recall"]; sum_f1 += m["f1"]; n_metric += 1

        except Exception as ex:
            pbar.set_postfix_str(f"skip:{Path(fp).name}")
            continue

    report = {
        "split": name,
        "n_graphs_total": len(files),
        "n_graphs_used": used,
        "avg_beam_length": (s_beam_len/max(1,used)),
        "interproc_step_share": (s_inter/max(1,s_steps)),
        "motif_match_share": (s_motif/max(1,s_steps)),
        "CFAM_mean": (sum(cfam_vals)/len(cfam_vals) if cfam_vals else None),
        "CCS_mean": (sum(ccs_vals)/len(ccs_vals) if ccs_vals else None),
        "precision": (sum_prec/max(1,n_metric) if n_metric else None),
        "recall": (sum_rec/max(1,n_metric) if n_metric else None),
        "f1": (sum_f1/max(1,n_metric) if n_metric else None),
    }
    return report

# ------------------- Reporting helpers -------------------
def write_text(path: Path, text: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)

def save_ckg_used(ckg, out_dir: Path):
    if not ckg: return None
    out_ckg = out_dir/"ckg_used.json"
    out_md  = out_dir/"ckg_used_report.md"
    write_text(out_ckg, json.dumps(ckg, indent=2, ensure_ascii=False))
    # MD
    lines=[]
    lines.append("# CKG Used\n")
    lines.append(f"- Graphs mined: {ckg.get('meta',{}).get('graphs_used','?')}")
    lines.append("\n## Top Motifs\n")
    motifs = ckg.get("motifs_topk", [])
    if motifs:
        lines.append("| # | Count | Relations |")
        lines.append("|---:|---:|---|")
        for i,m in enumerate(motifs[:20],1):
            rels = " → ".join(m.get("rels",[]))
            lines.append(f"| {i} | {m.get('count',0)} | {rels} |")
    else:
        lines.append("_No motifs in CKG._")
    write_text(out_md, "\n".join(lines))
    return str(out_ckg), str(out_md)

def save_beam_report(reports: Dict, out_dir: Path):
    md = out_dir/"beam_report.md"
    lines=[]
    lines.append("# Beam + ACC (+CKG) Evaluation\n")
    for k,v in reports.items():
        lines.append(f"## {k.capitalize()} split\n")
        lines.append(f"- Graphs total: {v['n_graphs_total']}")
        lines.append(f"- Graphs used:  {v['n_graphs_used']}")
        lines.append(f"- Avg beam length: {v['avg_beam_length']:.2f}")
        lines.append(f"- Inter-proc step share: {100*v['interproc_step_share']:.1f}%")
        lines.append(f"- Motif-match step share: {100*v['motif_match_share']:.1f}%")
        lines.append(f"- CFAM mean: {('%.4f'%v['CFAM_mean']) if v['CFAM_mean'] is not None else 'n/a'}")
        lines.append(f"- CCS mean:  {('%.4f'%v['CCS_mean']) if v['CCS_mean'] is not None else 'n/a'}")
        pr = v['precision']; rc=v['recall']; f1=v['f1']
        lines.append(f"- Precision/Recall/F1: "
                     f"{('%.3f'%pr if pr is not None else 'n/a')} / "
                     f"{('%.3f'%rc if rc is not None else 'n/a')} / "
                     f"{('%.3f'%f1 if f1 is not None else 'n/a')}")
        lines.append("")
    write_text(md, "\n".join(lines))
    return str(md)

# ------------------- Main -------------------
def main():
    ts = int(time.time())
    OUT_DIR = Path(f"out/beam_acc_eval_{ts}")
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    # Load model & CKG
    assert Path(CKPT_PATH).exists(), f"Checkpoint not found: {CKPT_PATH}"
    model, base_rel, add_summary = load_model(CKPT_PATH)
    ckg = load_ckg(CKG_PATH)

    # Calibrate threshold on a subset of VALID (if labels exist)
    valid_files = list_files(VALID_DIR, json_first=True)
    thresholds = calibrate_thresholds(model, valid_files, base_rel, add_summary, limit=min(400, len(valid_files))) \
                 if valid_files else {"default":0.25}

    # Evaluate VALID + TEST
    reports={}
    for split_name, d in [("valid", VALID_DIR), ("test", TEST_DIR)]:
        if not Path(d).exists():
            print(f"[skip] {split_name}: dir not found {d}")
            continue
        rep = eval_split(split_name, d, model, base_rel, add_summary, ckg, thresholds)
        reports[split_name]=rep
        print(f"[done] {split_name}: {rep}")

    # Save JSON summary
    meta = {"created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            "device": DEVICE,
            "ckpt": str(Path(CKPT_PATH).resolve()),
            "ckg":  (str(Path(CKG_PATH).resolve()) if CKG_PATH and Path(CKG_PATH).exists() else None),
            "config":{"beam":{"k":SEED_K,"width":BEAM_WIDTH,"max_hops":BEAM_MAX_HOPS,"alpha_node":ALPHA_NODE},
                      "lambda_prior": LAMBDA_PRIOR,
                      "acc":{"enforce":ENFORCE_ACC,"no_cycles":NO_CYCLES,"repeat_penalty":REPEAT_REL_PENALTY}}}
    out_json = OUT_DIR/"reports.json"
    write_text(out_json, json.dumps({"meta":meta,"reports":reports}, indent=2, ensure_ascii=False))

    # Save human-friendly MDs
    md_beam = save_beam_report(reports, OUT_DIR)
    ckg_json, ckg_md = save_ckg_used(ckg, OUT_DIR) if ckg else (None, None)

    print(f"\n[SAVED] reports → {OUT_DIR}")
    print(f"  - {out_json}")
    if md_beam: print(f"  - {md_beam}")
    if ckg_json: print(f"  - {ckg_json}")
    if ckg_md: print(f"  - {ckg_md}")

if __name__ == "__main__":
    main()


C:\Users\MSHUVO23\AppData\Local\Temp\ipykernel_31540\314252274.py:301: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location="cpu")


[env] torch=2.4.1+cu121 device=cuda


[calibrate]:   0%|          | 0/400 [00:00<?, ?graph/s]C:\Users\MSHUVO23\AppData\Local\Temp\ipykernel_31540\314252274.py:52: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  re

[done] valid: {'split': 'valid', 'n_graphs_total': 2906, 'n_graphs_used': 2906, 'avg_beam_length': 5.827598072952512, 'interproc_step_share': 0.07818566716444852, 'motif_match_share': 0.5407683345094113, 'CFAM_mean': 0.12732340876578066, 'CCS_mean': 2.1238581890810607e-07, 'precision': None, 'recall': None, 'f1': None}


[eval:test]: 100%|██████████| 2915/2915 [1:18:44<00:00,  1.62s/graph]  

[done] test: {'split': 'test', 'n_graphs_total': 2915, 'n_graphs_used': 2915, 'avg_beam_length': 5.8178387650085766, 'interproc_step_share': 0.07856028417509739, 'motif_match_share': 0.5391803798813974, 'CFAM_mean': 0.13016029964474324, 'CCS_mean': 2.071814705317219e-07, 'precision': None, 'recall': None, 'f1': None}

[SAVED] reports → out\beam_acc_eval_1761458865
  - out\beam_acc_eval_1761458865\reports.json
  - out\beam_acc_eval_1761458865\beam_report.md
  - out\beam_acc_eval_1761458865\ckg_used.json
  - out\beam_acc_eval_1761458865\ckg_used_report.md


TEST 2

In [1]:
# ===================== EVAL: Beam + CKG prior + ACC + CFAM/CCS_slice =====================
# Uses your trained model; evaluates on VALID and TEST; saves a single consolidated report.
# Labels are derived from `y` if present else from `sink_nodes` (if available).

import os, json, math, time, random
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
from collections import Counter, defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

# ---------- CONFIG ----------
CKPT_PATH   = r"out/cvul/1760989836/model.pt"       # your trained model
VALID_DIR   = r"Dataset/valid/hetero_ready_gcbert"  # folder of graphs
TEST_DIR    = r"Dataset/test/hetero_ready_gcbert"   # folder of graphs
CKG_PATH    = r"ckg/ckg.json"                       # optional; set to None to ignore CKG
OUT_DIR     = Path(f"beam_acc_ckg_eval/{int(time.time())}") # all outputs go here

# Beam & prior settings (align with training)
SEED_K        = 8
BEAM_WIDTH    = 24
BEAM_MAX_HOPS = 5
ALPHA_NODE    = 0.7
LAMBDA_PRIOR  = 0.15         # 0.1–0.3 is typical

# RNG / device
RNG_SEED = 23
USE_AMP  = True

# ---------- Device & Determinism ----------
def pick_device(min_free_mb=256):
    if not torch.cuda.is_available(): return "cpu"
    try:
        free,_ = torch.cuda.mem_get_info()
        return "cuda" if (free//(1024**2)) >= min_free_mb else "cpu"
    except:
        return "cuda"

DEVICE = pick_device()
random.seed(RNG_SEED)
torch.manual_seed(RNG_SEED)
if DEVICE=="cuda":
    torch.cuda.manual_seed_all(RNG_SEED)
print(f"[env] torch={torch.__version__} device={DEVICE}")

# ---------- IO utils ----------
def safe_load(p: str):
    pth = Path(p)
    if pth.suffix.lower()==".json":
        return json.loads(pth.read_text(encoding="utf-8"))
    return torch.load(pth, map_location="cpu")

def list_files(root: str, patterns=("*.json","*.pt")):
    root = Path(root)
    files=[]
    for pat in patterns:
        files.extend(sorted(root.glob(pat)))
    return [str(x) for x in files]

# ---------- Graph normalization (robust) ----------
RELATIONS_BASE = ["DFG","CFG","CALL","ARG2PARAM","RET2CALL","RET2LHS"]

def to_long_2(e):
    if e is None: return torch.zeros((2,0), dtype=torch.long)
    t = torch.as_tensor(e)
    if t.ndim==2 and t.shape[0]==2: return t.long().contiguous()
    if t.ndim==2 and t.shape[1]==2: return t.t().long().contiguous()
    if isinstance(e,(list,tuple)) and len(e)==2:
        s=torch.as_tensor(e[0]).view(-1).long()
        d=torch.as_tensor(e[1]).view(-1).long()
        return torch.stack([s,d], dim=0)
    if t.numel()==0: return torch.zeros((2,0), dtype=torch.long)
    raise RuntimeError("edge_index must be [2,E], [E,2], or (src,dst)")

def sanitize_edges(N:int, ei:torch.Tensor):
    if ei is None or ei.numel()==0: return torch.zeros((2,0), dtype=torch.long)
    s,d=ei
    m=(s>=0)&(s<N)&(d>=0)&(d<N)
    if m.any(): return torch.stack([s[m], d[m]], dim=0)
    return torch.zeros((2,0), dtype=torch.long)

def _first_2d_float(arr):
    t = torch.as_tensor(arr).float()
    if t.ndim==1: t=t.view(-1,1)
    return t

def coerce_x_any(obj: dict):
    # prefer precomputed GraphCodeBERT-like features
    for k in ["gcbert","gcb_x","x","features","node_features","feat","emb","x_text","x_num","x_dense","x_numeric"]:
        if k in obj and obj[k] is not None:
            t=torch.as_tensor(obj[k]).float()
            if t.ndim==1: t=t.view(-1,1)
            return t
    if "nodes" in obj and isinstance(obj["nodes"], list) and obj["nodes"]:
        rows=[]
        for nd in obj["nodes"]:
            if not isinstance(nd, dict): continue
            for k in ["gcbert","gcb_x","x","feat","emb","features"]:
                if k in nd and nd[k] is not None:
                    rows.append(torch.as_tensor(nd[k]).float().view(1,-1)); break
        if rows:
            d=max(r.size(1) for r in rows)
            rows=[F.pad(r,(0,d-r.size(1))) for r in rows]
            return torch.cat(rows, dim=0)
    return None

def build_edges_any(obj: dict, N: int, add_summary=True):
    E={r:torch.zeros((2,0),dtype=torch.long) for r in RELATIONS_BASE}
    if "edges" in obj and isinstance(obj["edges"], dict):
        for r in RELATIONS_BASE:
            if r in obj["edges"]:
                E[r]=sanitize_edges(N, to_long_2(obj["edges"][r]))
        if add_summary:
            base=E["DFG"]
            for r in ("ARG2PARAM","RET2CALL","RET2LHS"):
                if E[r].numel(): base=torch.cat([base,E[r]], dim=1)
            E["DFG_THIN"]=base
        return E
    for r in RELATIONS_BASE:
        for k in [r, f"{r}_edge_index", f"edge_index_{r}", f"{r.lower()}_edge_index", f"edge_index_{r.lower()}"]:
            if k in obj and obj[k] is not None:
                E[r]=sanitize_edges(N, to_long_2(obj[k])); break
    if add_summary:
        base=E["DFG"]
        for r in ("ARG2PARAM","RET2CALL","RET2LHS"):
            if E[r].numel(): base=torch.cat([base,E[r]], dim=1)
        E["DFG_THIN"]=base
    return E

def normalize_graph(obj, add_summary=True):
    # unwrap
    if isinstance(obj, dict) and "graph" in obj and isinstance(obj["graph"], dict):
        obj = obj["graph"]

    # torch_geometric.* object
    if "torch_geometric" in str(type(obj)) and not isinstance(obj, dict):
        # try hetero first
        node_stores=getattr(obj,"node_stores",None)
        edge_stores=getattr(obj,"edge_stores",None)
        if node_stores is not None and edge_stores is not None:
            cands=[]
            for st in node_stores:
                for nm in ["gcbert","gcb_x","x","x_text","x_num","features","feat","emb"]:
                    if hasattr(st,nm) and getattr(st,nm) is not None:
                        t=_first_2d_float(getattr(st,nm))
                        if t.numel()>0:
                            key = getattr(st,"_key", None) or getattr(st,"type", None) or "code"
                            cands.append((key,t)); break
            if cands:
                nt,x=max(cands, key=lambda kv: kv[1].size(0))
                N=x.size(0)
                E={r:torch.zeros((2,0),dtype=torch.long) for r in RELATIONS_BASE}
                for es in edge_stores:
                    ei=getattr(es,"edge_index",None)
                    if ei is None: continue
                    src_t=getattr(es,"src_type",None); dst_t=getattr(es,"dst_type",None)
                    rel = getattr(es,"edge_type",None) or getattr(es,"_key",None)
                    if isinstance(rel,(tuple,list)) and len(rel)==3: rel=str(rel[1])
                    rel = str(rel).upper().split("__")[-1] if rel is not None else None
                    if rel in E and (src_t is None or dst_t is None or (src_t==nt and dst_t==nt)):
                        E[rel]=sanitize_edges(N, to_long_2(ei))
                if add_summary:
                    base=E["DFG"]
                    for r in ("ARG2PARAM","RET2CALL","RET2LHS"):
                        if E[r].numel(): base=torch.cat([base,E[r]], dim=1)
                    E["DFG_THIN"]=base
                return {"x":x,"edges":E,"y":None,"source":"default"}

        # homogeneous
        if hasattr(obj,"x") and obj.x is not None:
            x=_first_2d_float(obj.x); N=x.size(0)
            E={r:torch.zeros((2,0),dtype=torch.long) for r in RELATIONS_BASE}
            if hasattr(obj,"edge_index"):
                E["DFG"]=sanitize_edges(N, to_long_2(obj.edge_index))
            if add_summary: E["DFG_THIN"]=E["DFG"]
            return {"x":x,"edges":E,"y":None,"source":"default"}
        return None

    # dict graph
    if isinstance(obj, dict):
        x = coerce_x_any(obj)
        if x is None or x.numel()==0: return None
        N = x.size(0)
        E = build_edges_any(obj, N, add_summary=add_summary)
        src = obj.get("source") or obj.get("dataset") or obj.get("origin") or "default"

        # labels: prefer y; else derive from sink_nodes
        y = None
        for k in ["y","label","labels","target","targets"]:
            if k in obj and obj[k] is not None:
                yy = torch.as_tensor(obj[k]).float().view(-1)
                if yy.numel()==N: y=yy; break
        if y is None and "sink_nodes" in obj and obj["sink_nodes"] is not None:
            y = torch.zeros(N).float()
            idx = torch.tensor(obj["sink_nodes"], dtype=torch.long)
            if idx.numel()>0:
                idx = idx.clamp_(0, N-1)
                y[idx] = 1.0

        g={"x":x,"edges":E,"y":y,"source":src}
        return g

    return None

# ---------- Model (match training) ----------
class GraphBlock(nn.Module):
    def __init__(self, hidden, relations):
        super().__init__()
        self.relations = relations
        self.lin_rel = nn.ModuleDict({r: nn.Linear(hidden,hidden,bias=False) for r in relations})
        self.lin_self= nn.Linear(hidden,hidden)
    def forward(self, h, E):
        H=h
        for r in self.relations:
            ei=E.get(r)
            if ei is None or ei.numel()==0: continue
            s,d=ei
            msg=self.lin_rel[r](H)
            agg=torch.zeros_like(H)
            agg.index_add_(0, d, msg[s])
            H=H+agg
        return self.lin_self(H)

class CausalVulNet(nn.Module):
    def __init__(self, hidden=64, layers=3, relations=None):
        super().__init__()
        if relations is None:
            relations = RELATIONS_BASE + ["DFG_THIN"]
        self.relations = list(relations)
        self.hidden = hidden
        self.layers = layers
        self.proj_cache = nn.ModuleDict()
        self.blocks = nn.ModuleList([GraphBlock(hidden, self.relations) for _ in range(layers)])
        self.node_head = nn.Linear(hidden,1)
        self.seed_head = nn.Linear(hidden,1)
        self.rel_gate  = nn.ParameterDict({r: nn.Parameter(torch.tensor(0.0)) for r in self.relations})
        self.edge_bilin= nn.Parameter(torch.empty(hidden, hidden)); nn.init.xavier_uniform_(self.edge_bilin)

    def _proj(self, D:int):
        k=str(D)
        if k not in self.proj_cache:
            layer=nn.Linear(D, self.hidden).to(next(self.parameters()).device)
            self.proj_cache[k]=layer
        return self.proj_cache[k]

    def encode(self, x, E):
        h=F.relu(self._proj(x.size(1))(x))
        for blk in self.blocks: h=F.elu(blk(h,E))
        return h

    def edge_scores(self, h, E):
        out={}
        for r,ei in E.items():
            if ei is None or ei.numel()==0:
                out[r]=torch.zeros((0,), device=h.device); continue
            s,d=ei
            hs=h[s] @ self.edge_bilin
            out[r]=(hs*h[d]).sum(dim=1) + self.rel_gate[r]
        return out

    def forward_full(self, x, E):
        seed_h=F.relu(self._proj(x.size(1))(x))
        seed_logit=self.seed_head(seed_h).squeeze(-1)
        h=self.encode(x,E)
        node_logit=self.node_head(h).squeeze(-1)
        edge_sc=self.edge_scores(h,E)
        return seed_logit, node_logit, h, edge_sc

def load_model(ckpt_path: str):
    ckpt = torch.load(ckpt_path, map_location="cpu")
    hidden    = ckpt.get("hidden", 64)
    layers    = ckpt.get("layers", 3)
    relations = ckpt.get("relations", RELATIONS_BASE + ["DFG_THIN"])
    model = CausalVulNet(hidden=hidden, layers=layers, relations=relations).to(DEVICE)
    model.load_state_dict(ckpt["state_dict"], strict=False)
    model.eval()
    base_rel = [r for r in relations if r!="DFG_THIN"]
    add_summary = ("DFG_THIN" in relations)
    return model, base_rel, add_summary

# ---------- Beam + CKG + ACC ----------
@dataclass
class BeamPath:
    score: float
    nodes: List[int]
    rels:  List[str]

def build_adj(E):
    adj_out={r:{} for r in E}; adj_in={r:{} for r in E}
    for r,ei in E.items():
        if ei is None or ei.numel()==0: continue
        s,d=ei; ss,dd=s.tolist(), d.tolist()
        for u,v in zip(ss,dd):
            adj_out[r].setdefault(u,[]).append(v)
            adj_in [r].setdefault(v,[]).append(u)
    return adj_out, adj_in

def pick_seeds(seed_logit, E, k):
    N=seed_logit.numel()
    deg=torch.zeros(N, device=seed_logit.device)
    for ei in E.values():
        if ei is None or ei.numel()==0: continue
        s,_=ei; deg.index_add_(0, s, torch.ones_like(s, dtype=deg.dtype))
    cand=torch.where(deg>0)[0]
    if cand.numel()==0: return torch.topk(seed_logit, k=min(k,N)).indices.tolist()
    k=min(k, cand.numel()); vals=seed_logit[cand]
    return cand[torch.topk(vals,k=k).indices].tolist()

def _edge_uv_scores(edge_sc, E, N:int):
    uv={}
    for r,ei in E.items():
        if ei is None or ei.numel()==0: uv[r]={}; continue
        s,d=ei; es=edge_sc[r].detach().float()
        mp={}
        for i in range(s.numel()):
            u=int(s[i]); v=int(d[i])
            if 0<=u<N and 0<=v<N:
                val=float(es[i].item())
                mp[(u,v)] = max(mp.get((u,v), val), val)
        uv[r]=mp
    return uv

def acc_admissible(next_node:int, cur_path: BeamPath, relation:str) -> Tuple[bool, List[str]]:
    """ACC filter: no revisit; relation is known; (hook for extra checks)."""
    notes=[]
    if relation not in (RELATIONS_BASE + ["DFG_THIN"]):
        return False, notes
    # no cycles
    if next_node in cur_path.nodes:
        return False, notes
    # record "ok" checks for transparency
    if relation=="CFG": notes.append("CFG-ok")
    elif relation in ("DFG","DFG_THIN"): notes.append("DFG-ok")
    elif relation in ("CALL","ARG2PARAM","RET2CALL","RET2LHS"): notes.append("Interproc-ok")
    else: notes.append("Relation-ok")
    return True, notes

def run_beam_with_ckg_acc(p, edge_sc, E, seeds, ckg, width=24, max_hops=5, alpha_node=0.7, lambda_prior=0.15):
    N=p.numel()
    adj_out, adj_in = build_adj(E)
    uv = _edge_uv_scores(edge_sc, E, N)

    # priors (log domain)
    ep = (ckg or {}).get("edge_prior_prob", {})
    bp = (ckg or {}).get("bigram_prob", {})
    motifs = (ckg or {}).get("motifs_topk", [])
    motif_tris = {tuple(m["rels"]) for m in motifs if "rels" in m}

    def logp_edge(r):      return math.log(max(float(ep.get(r, 1e-6)), 1e-9))
    def logp_bigram(a, b): return math.log(max(float(bp.get(a, {}).get(b, 1e-6)), 1e-9))

    def clog(x):           return float(torch.log(x.clamp(1e-9,1-1e-9)))

    beams=[BeamPath(clog(p[s]), [int(s)], []) for s in seeds if 0<=int(s)<N]
    if not beams: return []

    for _ in range(max_hops):
        nxt=[]
        for b in beams:
            u=b.nodes[-1]
            cand=[]
            for r in E.keys():
                # both out and in directions allowed
                for v in adj_out[r].get(u, []): cand.append((r,u,v))
                for v in adj_in [r].get(u, []): cand.append((r,v,u))
            if not cand:
                nxt.append(b); continue
            prev_r = b.rels[-1] if b.rels else None
            for (r,uu,vv) in cand:
                if not (0<=vv<N): continue
                ok, _ = acc_admissible(vv, b, r)
                if not ok: continue
                es  = uv.get(r,{}).get((uu,vv), 0.0)
                base = b.score + alpha_node*clog(p[vv]) + (1-alpha_node)*es
                # CKG priors
                if ckg:
                    base += lambda_prior * logp_edge(r)
                    if prev_r is not None:
                        base += lambda_prior * logp_bigram(prev_r, r)
                    # small extra nudge if the last triad matches a mined motif
                    if len(b.rels)>=2:
                        tri = (b.rels[-2], b.rels[-1], r)
                        if tri in motif_tris:
                            base += 0.10  # modest bonus for motif match
                nxt.append(BeamPath(base, b.nodes+[vv], b.rels+[r]))
        if not nxt: break
        nxt.sort(key=lambda x:x.score, reverse=True)
        beams = nxt[:width]
    return beams

# ---------- Metrics ----------
def f1_from_probs(p, y, thr=0.5):
    if y is None or y.numel()!=p.numel(): return None
    yb=(y>0.5); pb=(p>thr)
    tp=(pb&yb).sum().item(); fp=(pb&~yb).sum().item(); fn=(~pb&yb).sum().item()
    prec=tp/(tp+fp+1e-9); rec=tp/(tp+fn+1e-9); f1=2*prec*rec/(prec+rec+1e-9)
    return {"precision":prec,"recall":rec,"f1":f1}

def compute_CFAM(model, g_cpu, paths: List[BeamPath]):
    if not paths: return None
    x = g_cpu["x"].to(DEVICE).detach().requires_grad_(True)
    with torch.enable_grad():
        _, nl, _, _ = model.forward_full(x, {r:e.to(DEVICE) for r,e in g_cpu["edges"].items()})
        s = torch.sigmoid(nl).mean()
        s.backward()
        gn = x.grad.detach().abs().sum(dim=1)
    causal=set(n for bp in paths for n in bp.nodes if 0<=n<x.size(0))
    if not causal: return None
    mask=torch.zeros(x.size(0), dtype=torch.bool, device=gn.device)
    mask[torch.tensor(list(causal), device=gn.device)] = True
    num=gn[mask].sum().item(); den=gn.sum().item()+1e-9
    return num/den

def compute_CCS_slice(model, g_cpu, paths: List[BeamPath]):
    """Stronger CCS: measure change on beam nodes only (avoids graph-level dilution)"""
    if not paths: return None
    causal = sorted({n for bp in paths for n in bp.nodes})
    if not causal: return None
    x = g_cpu["x"].to(DEVICE)
    E = {r:e.to(DEVICE) for r,e in g_cpu["edges"].items()}
    with torch.no_grad():
        _, nl_full, _, _ = model.forward_full(x, E)
        p0 = torch.sigmoid(nl_full[torch.tensor(causal, device=x.device)]).mean().item()
    x_cf = x.clone()
    x_cf[torch.tensor(causal, device=x.device)] = 0.0     # stronger local cut
    with torch.no_grad():
        _, nl_cf, _, _ = model.forward_full(x_cf, E)
        p1 = torch.sigmoid(nl_cf[torch.tensor(causal, device=x.device)]).mean().item()
    return (p0 - p1) ** 2

# ---------- Calibration (on VALID only) ----------
def calibrate_thresholds(model, files_valid, add_summary=True):
    buf_by_src = defaultdict(lambda: {"p":[], "y":[]})
    pbar = tqdm(files_valid, desc="[calibrate]", unit="graph")
    for fp in pbar:
        try:
            g_cpu = normalize_graph(safe_load(fp), add_summary=add_summary)
            if g_cpu is None or g_cpu["x"] is None or g_cpu["x"].numel()==0: continue
            if g_cpu["y"] is None: continue  # need labels
            x = g_cpu["x"].to(DEVICE)
            E = {r:e.to(DEVICE) for r,e in g_cpu["edges"].items()}
            with torch.no_grad():
                _, nl, _, _ = model.forward_full(x, E)
                p = torch.sigmoid(nl).detach().cpu()
            src = g_cpu.get("source","default")
            buf_by_src[src]["p"].append(p)
            buf_by_src[src]["y"].append(g_cpu["y"].cpu().float().view(-1))
        except Exception:
            continue
    thrs={}
    for src,buf in buf_by_src.items():
        if not buf["p"]: continue
        P = torch.cat(buf["p"]); Y = torch.cat(buf["y"])
        best_thr, best_f = 0.25, 0.0
        for thr in [i/100 for i in range(5,96,1)]:
            m=f1_from_probs(P, Y, thr)
            if m and m["f1"]>best_f:
                best_thr, best_f = thr, m["f1"]
        thrs[src]=best_thr
    if not thrs: thrs={"default":0.25}
    return thrs

# ---------- Eval loop ----------
def eval_split(split_name:str, data_dir:str, model, add_summary:bool, ckg:Optional[dict], thresholds:Dict[str,float]):
    files = list_files(data_dir)
    if not files:
        return {"n_graphs":0}

    interproc_set = {"CALL","ARG2PARAM","RET2CALL","RET2LHS"}

    n_used=0
    s_prec=s_rec=s_f1=0.0
    cfam_vals=[]; ccs_vals=[]
    avg_beam_len=[]; inter_ratio=[]
    motif_hit_count=0; motif_total=0

    motifs = (ckg or {}).get("motifs_topk", [])
    motif_tris = {tuple(m["rels"]) for m in motifs if "rels" in m}

    pbar = tqdm(files, desc=f"[eval:{split_name}]", unit="graph")
    for fp in pbar:
        try:
            g_cpu = normalize_graph(safe_load(fp), add_summary=add_summary)
            if g_cpu is None or g_cpu["x"] is None or g_cpu["x"].numel()==0: 
                continue
            x = g_cpu["x"].to(DEVICE)
            E = {r:e.to(DEVICE) for r,e in g_cpu["edges"].items()}
            with torch.no_grad():
                sd, nl, h, es = model.forward_full(x, E)
                seeds = pick_seeds(sd.detach(), E, SEED_K)
                p = torch.sigmoid(nl)
                with torch.autocast("cuda", enabled=(USE_AMP and DEVICE=="cuda")):
                    beams = run_beam_with_ckg_acc(p, es, E, seeds, ckg, width=BEAM_WIDTH,
                                                  max_hops=BEAM_MAX_HOPS, alpha_node=ALPHA_NODE,
                                                  lambda_prior=LAMBDA_PRIOR)
            if not beams: 
                continue

            # beam stats
            lens=[len(bp.nodes)-1 for bp in beams]
            avg_beam_len.append(sum(lens)/(len(lens) or 1))
            # inter-proc %
            inter_steps=0; total_steps=0
            for bp in beams:
                for r in bp.rels:
                    total_steps+=1
                    if r in interproc_set: inter_steps+=1
                # motif coverage
                for i in range(2, len(bp.rels)):
                    tri = (bp.rels[i-2], bp.rels[i-1], bp.rels[i])
                    motif_total += 1
                    if tri in motif_tris: motif_hit_count += 1
            inter_ratio.append(inter_steps/(total_steps or 1))

            # metrics (needs thresholds & labels)
            y=g_cpu.get("y")
            if y is not None and y.numel()==p.numel():
                thr = thresholds.get(g_cpu.get("source","default"), thresholds.get("default",0.25))
                m = f1_from_probs(p.detach().cpu(), y.cpu().float().view(-1), thr)
                if m:
                    s_prec+=m["precision"]; s_rec+=m["recall"]; s_f1+=m["f1"]

            # CFAM / CCS on top beam (ckg+acc version)
            topk = beams[:1]
            cf = compute_CFAM(model, g_cpu, topk)
            cc = compute_CCS_slice(model, g_cpu, topk)
            if cf is not None: cfam_vals.append(cf)
            if cc is not None: ccs_vals.append(cc)

            n_used+=1

        except Exception as ex:
            pbar.set_postfix_str(f"skip: {Path(fp).name}")

    rep = {
        "split": split_name,
        "n_graphs_used": n_used,
        "beam": {
            "avg_length_edges": (sum(avg_beam_len)/len(avg_beam_len) if avg_beam_len else None),
            "interproc_ratio":  (sum(inter_ratio)/len(inter_ratio) if inter_ratio else None),
            "motif_hit_rate":   (motif_hit_count/(motif_total or 1))
        },
        "metrics": {
            "precision": (s_prec/n_used if n_used and s_prec>0 else None),
            "recall":    (s_rec/n_used if n_used and s_rec>0 else None),
            "f1":        (s_f1/n_used if n_used and s_f1>0 else None),
            "CFAM_mean": (sum(cfam_vals)/len(cfam_vals) if cfam_vals else None),
            "CCS_mean":  (sum(ccs_vals)/len(ccs_vals) if ccs_vals else None)
        }
    }
    return rep

# ---------- Friendly markdown summaries ----------
def write_beam_report_md(out_path: str, valid_rep: dict, test_rep: dict):
    lines=[]
    lines.append("# Beam + CKG + ACC Evaluation\n")
    for name, rep in [("VALID", valid_rep), ("TEST", test_rep)]:
        if not rep or rep.get("n_graphs_used",0)==0:
            lines.append(f"## {name}\nNo usable graphs.\n"); continue
        b=rep["beam"]; m=rep["metrics"]
        lines.append(f"## {name}\n")
        lines.append(f"- Graphs used: **{rep['n_graphs_used']}**")
        lines.append(f"- Avg beam length (edges): **{b['avg_length_edges']:.2f}**" if b["avg_length_edges"] is not None else "- Avg beam length: n/a")
        lines.append(f"- Inter-procedural hop ratio: **{100*(b['interproc_ratio'] or 0):.1f}%**" if b["interproc_ratio"] is not None else "- Inter-procedural hop ratio: n/a")
        lines.append(f"- Motif hit rate: **{100*(b['motif_hit_rate'] or 0):.1f}%**")
        lines.append(f"- Precision: **{m['precision']:.3f}**, Recall: **{m['recall']:.3f}**, F1: **{m['f1']:.3f}**"
                     if m["precision"] is not None else "- Precision/Recall/F1: n/a (labels absent)")
        lines.append(f"- CFAM (mean): **{m['CFAM_mean']:.3f}**" if m["CFAM_mean"] is not None else "- CFAM: n/a")
        lines.append(f"- CCS_slice (mean): **{m['CCS_mean']:.3f}**\n" if m["CCS_mean"] is not None else "- CCS_slice: n/a\n")
    Path(out_path).write_text("\n".join(lines), encoding="utf-8")

def write_ckg_used_md(out_path: str, ckg: Optional[dict]):
    lines=["# CKG Prior Used\n"]
    if not ckg:
        lines.append("_No CKG provided._")
        Path(out_path).write_text("\n".join(lines), encoding="utf-8"); return
    lines.append(f"- Relations: {', '.join(ckg.get('relations', []))}")
    lines.append(f"- Motifs: {len(ckg.get('motifs_topk', []))} tri-grams\n")
    lines.append("## Top 10 Motifs\n")
    lines.append("| # | Count | Motif |")
    lines.append("|---:|---:|---|")
    for i,m in enumerate(ckg.get("motifs_topk", [])[:10], 1):
        lines.append(f"| {i} | {m.get('count',0)} | {' → '.join(m.get('rels',[]))} |")
    Path(out_path).write_text("\n".join(lines), encoding="utf-8")

# ---------- MAIN ----------
def main():
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    # load model
    assert os.path.exists(CKPT_PATH), f"Checkpoint not found: {CKPT_PATH}"
    model, base_rel, add_summary = load_model(CKPT_PATH)
    print(f"[model] hidden={model.hidden} layers={model.layers} relations={model.relations}")
    if DEVICE=="cuda":
        try: print("GPU:", torch.cuda.get_device_name(0))
        except: pass

    # load CKG (optional)
    ckg = None
    if CKG_PATH and os.path.exists(CKG_PATH):
        ckg = json.loads(Path(CKG_PATH).read_text(encoding="utf-8"))
        # persist the exact prior used
        Path(OUT_DIR/"ckg_used.json").write_text(json.dumps(ckg, indent=2), encoding="utf-8")
        write_ckg_used_md(str(OUT_DIR/"ckg_used_report.md"), ckg)
        print(f"[ckg] loaded {CKG_PATH} | motifs={len(ckg.get('motifs_topk',[]))}")
    else:
        print("[ckg] none provided (running without prior)")

    # files
    valid_files = list_files(VALID_DIR)
    test_files  = list_files(TEST_DIR)

    # Calibrate thresholds on VALID (only graphs that have labels/sinks)
    thrs = calibrate_thresholds(model, valid_files, add_summary=add_summary)
    Path(OUT_DIR/"thresholds.json").write_text(json.dumps(thrs, indent=2), encoding="utf-8")
    print(f"[calibration] thresholds_by_source: {thrs}")

    # Evaluate VALID and TEST
    rep_valid = eval_split("valid", VALID_DIR, model, add_summary, ckg, thrs)
    rep_test  = eval_split("test",  TEST_DIR,  model, add_summary, ckg, thrs)

    # Save JSON summary
    reports = {
        "config": {
            "ckpt": str(Path(CKPT_PATH).resolve()),
            "valid_dir": str(Path(VALID_DIR).resolve()),
            "test_dir":  str(Path(TEST_DIR).resolve()),
            "ckg": (str(Path(CKG_PATH).resolve()) if CKG_PATH and os.path.exists(CKG_PATH) else None),
            "beam": {"seed_k":SEED_K,"width":BEAM_WIDTH,"max_hops":BEAM_MAX_HOPS,"alpha_node":ALPHA_NODE,"lambda_prior":LAMBDA_PRIOR}
        },
        "reports": {
            "valid": rep_valid,
            "test":  rep_test
        }
    }
    Path(OUT_DIR/"reports.json").write_text(json.dumps(reports, indent=2), encoding="utf-8")

    # Markdown summaries
    write_beam_report_md(str(OUT_DIR/"beam_report.md"), rep_valid, rep_test)

    print(f"[SAVED] reports → {OUT_DIR}")

if __name__ == "__main__":
    main()


C:\Users\MSHUVO23\AppData\Local\Temp\ipykernel_29244\3132525413.py:274: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location="cpu")


[env] torch=2.4.1+cu121 device=cuda
[model] hidden=64 layers=3 relations=['DFG', 'CFG', 'CALL', 'ARG2PARAM', 'RET2CALL', 'RET2LHS', 'DFG_THIN']
GPU: NVIDIA GeForce RTX 4070 Laptop GPU
[ckg] loaded ckg/ckg.json | motifs=20


[calibrate]:   0%|          | 0/2906 [00:00<?, ?graph/s]C:\Users\MSHUVO23\AppData\Local\Temp\ipykernel_29244\3132525413.py:55: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  

[calibration] thresholds_by_source: {'default': 0.25}


[eval:test]: 100%|██████████| 2915/2915 [1:15:53<00:00,  1.56s/graph] 

[SAVED] reports → beam_acc_ckg_eval\1761469116


TEST 3

In [2]:

import os, json, math, time, random
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional
from collections import Counter, defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

# -------------------- CONFIG (edit these) --------------------
CKPT_PATH = r"out/cvul/1760989836/model.pt"
VALID_DIR = r"Dataset/valid/hetero_ready_gcbert"
TEST_DIR  = r"Dataset/test/hetero_ready_gcbert"
CKG_PATH  = r"ckg/ckg.json"
OUT_DIR   = r"out/eval_full"   # single folder for all outputs

# Beam and prior weights (match training where applicable)
SEED_K        = 8
BEAM_WIDTH    = 24
BEAM_MAX_HOPS = 5
ALPHA_NODE    = 0.7            # node-vs-edge blending
LAMBDA_EDGE   = 0.15           # weight for edge prior P(r)
LAMBDA_BIGRAM = 0.15           # weight for bigram prior P(r_t | r_{t-1})
LAMBDA_MOTIF  = 0.20           # extra weight if (r_{t-2}, r_{t-1}, r_t) hits mined tri-gram motif

# Inter-procedural emphasis in CKG (boost helpful motifs; damp trivial CFG/CALL loops)
INTERPROC_SET = {"ARG2PARAM","RET2CALL","RET2LHS","DFG_THIN"}
MOTIF_INTERPROC_BOOST = 2.0
MOTIF_TRIVIAL_DAMP    = 0.5

# Eval controls
RNG_SEED        = 23
USE_AMP         = True
EVAL_LIMIT_VALID= None     # None = evaluate all graphs; or set an int for quick run
EVAL_LIMIT_TEST = None
CONF_GRAPH_MINLEN = 2      # require top beam length >= 2 for CFAM/CCS
# -------------------- Device --------------------
def pick_device(min_free_mb=256):
    if not torch.cuda.is_available(): return "cpu"
    try:
        free,_ = torch.cuda.mem_get_info()
        return "cuda" if (free//(1024**2)) >= min_free_mb else "cpu"
    except:
        return "cuda"
DEVICE = pick_device()
print(f"[env] torch={torch.__version__} device={DEVICE}")
random.seed(RNG_SEED); torch.manual_seed(RNG_SEED)
if DEVICE=="cuda":
    try: torch.cuda.manual_seed_all(RNG_SEED)
    except: pass

# -------------------- IO helpers --------------------
def safe_load(p):
    p = Path(p)
    if p.suffix.lower() == ".json":
        return json.loads(p.read_text(encoding="utf-8"))
    return torch.load(p, map_location="cpu")

def list_files(root_dir, patterns=("*.json","*.pt")):
    root = Path(root_dir)
    files=[]
    for pat in patterns:
        files.extend(sorted(root.glob(pat)))
    return files

# -------------------- Normalization (robust) --------------------
def _first_2d_float(arr):
    t = torch.as_tensor(arr).float()
    if t.ndim == 1: t = t.view(-1, 1)
    return t

def to_long_2(e):
    if e is None: return torch.zeros((2,0), dtype=torch.long)
    t=torch.as_tensor(e)
    if t.ndim==2 and t.shape[0]==2: return t.long().contiguous()
    if t.ndim==2 and t.shape[1]==2: return t.t().long().contiguous()
    if isinstance(e,(list,tuple)) and len(e)==2:
        s=torch.as_tensor(e[0]).view(-1).long()
        d=torch.as_tensor(e[1]).view(-1).long()
        return torch.stack([s,d], dim=0)
    if t.numel()==0: return torch.zeros((2,0), dtype=torch.long)
    raise RuntimeError("edge_index must be [2,E], [E,2], or (src,dst)")

def sanitize_edges(N:int, ei:torch.Tensor):
    if ei is None or ei.numel()==0: return torch.zeros((2,0), dtype=torch.long)
    s,d=ei
    m=(s>=0)&(s<N)&(d>=0)&(d<N)
    if m.any(): return torch.stack([s[m], d[m]], dim=0)
    return torch.zeros((2,0), dtype=torch.long)

def _get_store_feat(store):
    cand = ["gcbert","gcb_x","x","x_text","x_num","features","feat","emb"]
    for nm in cand:
        if hasattr(store, nm) and getattr(store, nm) is not None:
            t = _first_2d_float(getattr(store, nm))
            if t.numel() > 0: return t
    if hasattr(store, "__dict__"):
        for nm in cand:
            if nm in store.__dict__ and store.__dict__[nm] is not None:
                t = _first_2d_float(store.__dict__[nm])
                if t.numel() > 0: return t
    return None

def normalize_hetero(obj, RELATIONS, ADD_SUMMARY_EDGES=True):
    node_stores = getattr(obj, "node_stores", None)
    edge_stores = getattr(obj, "edge_stores", None)
    if node_stores is None or edge_stores is None:
        return None

    candidates=[]
    for st in node_stores:
        key = getattr(st, "_key", None) or getattr(st, "type", None) or "code"
        feat = _get_store_feat(st)
        if feat is not None and feat.numel() > 0:
            candidates.append((key, feat))
    if not candidates: return None
    nt, x = max(candidates, key=lambda kv: kv[1].size(0))
    N = x.size(0)

    E = {r: torch.zeros((2,0), dtype=torch.long) for r in RELATIONS}
    for es in edge_stores:
        ei = getattr(es, "edge_index", None)
        if ei is None: continue
        src_t = getattr(es, "src_type", None)
        dst_t = getattr(es, "dst_type", None)
        if (src_t is not None and dst_t is not None) and not (src_t == nt and dst_t == nt):
            continue
        rel = getattr(es, "edge_type", None) or getattr(es, "_key", None)
        if isinstance(rel, (tuple, list)) and len(rel) == 3:
            rel = str(rel[1])
        rel = str(rel).upper().split("__")[-1] if rel is not None else None
        if rel in E:
            E[rel] = sanitize_edges(N, to_long_2(ei))

    if ADD_SUMMARY_EDGES:
        base = E.get("DFG", torch.zeros((2,0), dtype=torch.long))
        for r in ("ARG2PARAM", "RET2CALL", "RET2LHS"):
            if r in E and E[r].numel() > 0:
                base = torch.cat([base, E[r]], dim=1)
        E["DFG_THIN"] = base

    # labels (none by default for hetero)
    return {"x": x, "edges": E, "y": None, "source": "default"}

def coerce_x_any(obj):
    for k in ["gcbert","gcb_x","x","features","node_features","feat","emb","x_text","x_num","x_dense","x_numeric"]:
        if isinstance(obj, dict) and k in obj and obj[k] is not None:
            t=torch.as_tensor(obj[k]).float()
            if t.ndim==1: t=t.view(-1,1)
            return t
    if isinstance(obj, dict) and "nodes" in obj and isinstance(obj["nodes"], list) and obj["nodes"]:
        rows=[]
        for nd in obj["nodes"]:
            if not isinstance(nd, dict): continue
            for k in ["gcbert","gcb_x","x","feat","emb","features"]:
                if k in nd and nd[k] is not None:
                    rows.append(torch.as_tensor(nd[k]).float().view(1,-1)); break
        if rows:
            d=max(r.size(1) for r in rows)
            rows=[F.pad(r,(0,d-r.size(1))) for r in rows]
            return torch.cat(rows, dim=0)
    return None

def build_edges_any(obj, N, RELATIONS, ADD_SUMMARY_EDGES=True):
    E={r:torch.zeros((2,0),dtype=torch.long) for r in RELATIONS}
    if isinstance(obj, dict) and "edges" in obj and isinstance(obj["edges"], dict):
        for r in RELATIONS:
            if r in obj["edges"]:
                E[r]=sanitize_edges(N, to_long_2(obj["edges"][r]))
        if ADD_SUMMARY_EDGES:
            base=E.get("DFG", torch.zeros((2,0),dtype=torch.long))
            for r in ("ARG2PARAM","RET2CALL","RET2LHS"):
                if E[r].numel(): base=torch.cat([base,E[r]], dim=1)
            E["DFG_THIN"]=base
        return E
    for r in RELATIONS:
        for k in [r, f"{r}_edge_index", f"edge_index_{r}", f"{r.lower()}_edge_index", f"edge_index_{r.lower()}"]:
            if isinstance(obj, dict) and k in obj:
                E[r]=sanitize_edges(N, to_long_2(obj[k])); break
    if ADD_SUMMARY_EDGES:
        base=E.get("DFG", torch.zeros((2,0),dtype=torch.long))
        for r in ("ARG2PARAM","RET2CALL","RET2LHS"):
            if E[r].numel(): base=torch.cat([base,E[r]], dim=1)
        E["DFG_THIN"]=base
    return E

def normalize_graph(obj, RELATIONS, ADD_SUMMARY_EDGES=True):
    # unwrap
    if isinstance(obj, dict) and "graph" in obj and isinstance(obj["graph"], dict):
        obj=obj["graph"]

    # torch_geometric (hetero or homogeneous)
    if "torch_geometric" in str(type(obj)) and not isinstance(obj, dict):
        g_het = normalize_hetero(obj, RELATIONS, ADD_SUMMARY_EDGES)
        if g_het is not None: return g_het
        if hasattr(obj, "x") and obj.x is not None:
            x = _first_2d_float(obj.x); N=x.size(0)
            E = build_edges_any({"edge_index": getattr(obj,"edge_index", None)}, N, RELATIONS, ADD_SUMMARY_EDGES)
            if "DFG" not in E and hasattr(obj,"edge_index"):
                E["DFG"] = sanitize_edges(N, to_long_2(obj.edge_index))
            if ADD_SUMMARY_EDGES and "DFG_THIN" not in E and "DFG" in E:
                E["DFG_THIN"]=E["DFG"]
            return {"x":x, "edges":E, "y":None, "source":"default"}
        return None

    # json or dict-like
    if isinstance(obj, dict):
        x = coerce_x_any(obj)
        if x is None or x.numel()==0:
            return None
        N = x.size(0)
        E = build_edges_any(obj, N, RELATIONS, ADD_SUMMARY_EDGES)

        # ----- labels: primary + fallback from sinks -----
        y=None
        for key in ("y","labels","targets","target","vulnerable"):
            if key in obj:
                yy = torch.as_tensor(obj[key]).float().view(-1)
                if yy.numel() == N: y = yy; break
        if y is None:
            for key in ("sink_nodes","sinks"):
                if key in obj and isinstance(obj[key], (list,tuple)):
                    yy = torch.zeros(N, dtype=torch.float32)
                    for i in obj[key]:
                        j = int(i)
                        if 0 <= j < N: yy[j] = 1.0
                    # only set if at least one positive
                    if yy.sum().item() > 0:
                        y = yy
                    break

        src = obj.get("source") or obj.get("dataset") or obj.get("origin") or "default"
        return {"x":x, "edges":E, "y":y, "source":src}

    return None

def iter_graphs(root_dir, RELATIONS, ADD_SUMMARY_EDGES=True, limit=None, patterns=("*.json","*.pt")):
    files = list_files(root_dir, patterns)
    if limit is not None:
        files = files[:limit]
    for fp in tqdm(files, desc=f"[load] {root_dir}", unit="graph"):
        try:
            obj = safe_load(fp)
            g = normalize_graph(obj, RELATIONS, ADD_SUMMARY_EDGES)
            if g is None or g["x"] is None or g["x"].numel()==0:
                print(f"[skip] {fp} — no usable node features")
                continue
            g["path"] = str(fp)
            yield g
        except Exception as ex:
            print(f"[skip] {fp} error: {ex}")

# -------------------- Model --------------------
class GraphBlock(nn.Module):
    def __init__(self, hidden, relations):
        super().__init__()
        self.relations=relations
        self.lin_rel=nn.ModuleDict({r:nn.Linear(hidden,hidden,bias=False) for r in relations})
        self.lin_self=nn.Linear(hidden,hidden)
    def forward(self, h, E):
        H=h
        for r in self.relations:
            ei=E.get(r)
            if ei is None or ei.numel()==0: continue
            s,d=ei
            msg=self.lin_rel[r](H)
            agg=torch.zeros_like(H)
            agg.index_add_(0, d, msg[s])
            H=H+agg
        return self.lin_self(H)

class CausalVulNet(nn.Module):
    def __init__(self, hidden, layers, relations, add_summary=True):
        super().__init__()
        self.relations = list(relations) + (["DFG_THIN"] if add_summary and "DFG_THIN" not in relations else [])
        self.proj_cache = nn.ModuleDict()
        self.blocks = nn.ModuleList([GraphBlock(hidden, self.relations) for _ in range(layers)])
        self.node_head = nn.Linear(hidden,1)
        self.seed_head = nn.Linear(hidden,1)
        self.rel_gate  = nn.ParameterDict({r: nn.Parameter(torch.tensor(0.0)) for r in self.relations})
        self.edge_bilin= nn.Parameter(torch.empty(hidden, hidden)); nn.init.xavier_uniform_(self.edge_bilin)
        self.hidden=hidden; self.layers=layers
    def _proj(self, D:int):
        k=str(D)
        if k not in self.proj_cache:
            layer=nn.Linear(D, self.hidden).to(next(self.parameters()).device)
            self.proj_cache[k]=layer
        return self.proj_cache[k]
    def encode(self, x, E):
        h=F.relu(self._proj(x.size(1))(x))
        for blk in self.blocks: h=F.elu(blk(h,E))
        return h
    def edge_scores(self, h, E):
        out={}
        for r,ei in E.items():
            if ei is None or ei.numel()==0:
                out[r]=torch.zeros((0,), device=h.device); continue
            s,d=ei
            hs=h[s] @ self.edge_bilin
            out[r]=(hs*h[d]).sum(dim=1) + self.rel_gate[r]
        return out
    def forward_full(self, x, E):
        seed_h=F.relu(self._proj(x.size(1))(x))
        seed_logit=self.seed_head(seed_h).squeeze(-1)
        h=self.encode(x,E)
        node_logit=self.node_head(h).squeeze(-1)
        edge_sc=self.edge_scores(h,E)
        return seed_logit, node_logit, h, edge_sc

def load_model(ckpt_path:str):
    ckpt = torch.load(ckpt_path, map_location="cpu")
    hidden    = ckpt.get("hidden", 64)
    layers    = ckpt.get("layers", 3)
    relations = ckpt.get("relations", ["DFG","CFG","CALL","ARG2PARAM","RET2CALL","RET2LHS","DFG_THIN"])
    add_summary = ("DFG_THIN" in relations)
    model = CausalVulNet(hidden=hidden, layers=layers, relations=relations, add_summary=add_summary).to(DEVICE)
    model.load_state_dict(ckpt["state_dict"], strict=False)
    model.eval()
    base_rel = [r for r in relations if r!="DFG_THIN"]
    return model, base_rel, add_summary

# -------------------- Beam + CKG + ACC --------------------
@dataclass
class BeamPath:
    score: float
    nodes: List[int]
    rels:  List[str]

def build_adj(E):
    adj_out={r:{} for r in E}; adj_in={r:{} for r in E}
    for r,ei in E.items():
        if ei is None or ei.numel()==0: continue
        s,d=ei; ss,dd=s.tolist(), d.tolist()
        for u,v in zip(ss,dd):
            adj_out[r].setdefault(u,[]).append(v)
            adj_in [r].setdefault(v,[]).append(u)
    return adj_out, adj_in

def pick_seeds(seed_logit, E, k):
    N=seed_logit.numel()
    deg=torch.zeros(N, device=seed_logit.device)
    for ei in E.values():
        if ei is None or ei.numel()==0: continue
        s,_=ei; deg.index_add_(0, s, torch.ones_like(s, dtype=deg.dtype))
    cand=torch.where(deg>0)[0]
    if cand.numel()==0: return torch.topk(seed_logit, k=min(k,N)).indices.tolist()
    k=min(k, cand.numel()); vals=seed_logit[cand]
    return cand[torch.topk(vals,k=k).indices].tolist()

def _edge_uv_scores(edge_sc, E, N:int):
    uv={}
    for r,ei in E.items():
        if ei is None or ei.numel()==0: uv[r]={}; continue
        s,d=ei; es=edge_sc[r].detach().float()
        mp={}
        for i in range(s.numel()):
            u=int(s[i]); v=int(d[i])
            if 0<=u<N and 0<=v<N:
                val=float(es[i].item())
                mp[(u,v)] = max(mp.get((u,v), val), val)
        uv[r]=mp
    return uv

# Simple ACC checks: no cycles; disallow immediate backtracking; allow only existing edges (already enforced)
def acc_ok(prev_nodes:List[int], u:int, v:int, r:str, adj_out, adj_in)->bool:
    if v in prev_nodes:             # no revisits
        return False
    if len(prev_nodes)>=2 and v==prev_nodes[-2]:  # no backtrack
        return False
    # Could extend with stricter per-relation constraints if your data encodes them
    return True

# Build tri-gram probability table from CKG motifs_topk (counts → probs)
def build_trigram_probs(ckg:Dict)->Dict[Tuple[str,str,str], float]:
    motifs = ckg.get("motifs_topk", [])
    if not motifs: return {}
    total = sum(int(m.get("count",1)) for m in motifs) or 1
    tri = {}
    for m in motifs:
        rels = m.get("rels", [])
        if len(rels)==3:
            tri[(rels[0],rels[1],rels[2])] = max(1e-9, int(m.get("count",1)) / total)
    return tri

def motif_multiplier(rel_seq:Tuple[str,str,str])->float:
    a,b,c = rel_seq
    has_inter = any(r in INTERPROC_SET for r in rel_seq)
    only_trivial = all(r in {"CFG","CALL"} for r in rel_seq)
    if has_inter:     return MOTIF_INTERPROC_BOOST
    if only_trivial:  return MOTIF_TRIVIAL_DAMP
    return 1.0

def run_beam_ckg_acc(p, edge_sc, E, seeds, ckg,
                     width=24, max_hops=5, alpha_node=0.7,
                     lambda_edge=0.15, lambda_bigram=0.15, lambda_motif=0.20):
    N=p.numel()
    adj_out, adj_in = build_adj(E)
    uv = _edge_uv_scores(edge_sc, E, N)

    # priors (log domain)
    ep = ckg.get("edge_prior_prob", {})        # P(r)
    bp = ckg.get("bigram_prob", {})            # P(r_t | r_{t-1})
    tp = build_trigram_probs(ckg)              # P(r_{t-2},r_{t-1},r_t) from motifs_topk

    def logp_edge(r):      return math.log(max(float(ep.get(r, 1e-6)), 1e-9))
    def logp_bigram(a,b):  return math.log(max(float(bp.get(a, {}).get(b, 1e-6)), 1e-9))
    def logp_trigram(a,b,c):
        base = max(1e-9, tp.get((a,b,c), 1e-9))
        return math.log(base) * motif_multiplier((a,b,c))
    def clog(x):           return float(torch.log(x.clamp(1e-9,1-1e-9)))

    beams=[BeamPath(clog(p[s]), [int(s)], []) for s in seeds if 0<=int(s)<N]
    if not beams: return []
    out=[]
    for _ in range(max_hops):
        nxt=[]
        for b in beams:
            u=b.nodes[-1]
            cand=[]
            for r in E.keys():
                for v in adj_out[r].get(u, []): cand.append((r,u,v))
                for v in adj_in [r].get(u, []): cand.append((r,v,u))
            if not cand: out.append(b); continue
            for (r,uu,vv) in cand:
                if not (0<=vv<N): continue
                if not acc_ok(b.nodes, uu, vv, r, adj_out, adj_in):  # ACC gate
                    continue
                es = uv.get(r,{}).get((uu,vv), 0.0)
                sc = b.score + alpha_node*clog(p[vv]) + (1-alpha_node)*es
                sc += lambda_edge * logp_edge(r)
                if b.rels:
                    sc += lambda_bigram * logp_bigram(b.rels[-1], r)
                if len(b.rels)>=2:
                    sc += lambda_motif  * logp_trigram(b.rels[-2], b.rels[-1], r)
                nxt.append(BeamPath(sc, b.nodes+[vv], b.rels+[r]))
        if not nxt: break
        nxt.sort(key=lambda x:x.score, reverse=True)
        beams = nxt[:width]
    out.extend(beams); out.sort(key=lambda x:x.score, reverse=True)
    return out[:width]

# -------------------- Metrics --------------------
def f1_from_probs(p, y, thr=0.5):
    if y is None or y.numel()!=p.numel(): return None
    yb=(y>0.5); pb=(p>thr)
    tp=(pb&yb).sum().item(); fp=(pb&~yb).sum().item(); fn=(~pb&yb).sum().item()
    prec=tp/(tp+fp+1e-9); rec=tp/(tp+fn+1e-9); f1=2*prec*rec/(prec+rec+1e-9)
    return {"precision":prec,"recall":rec,"f1":f1}

@torch.no_grad()
def calibrate_thresholds(model, graphs, base_rel, add_summary, limit=None):
    per_source = {}
    files = graphs if isinstance(graphs, list) else list(graphs)
    if limit is not None: files = files[:limit]
    pbar = tqdm(files, desc="[calibrate]", unit="graph")
    for g_cpu in pbar:
        g = {"x": g_cpu["x"].to(DEVICE),
             "edges": {r:e.to(DEVICE) for r,e in g_cpu["edges"].items()},
             "y": (g_cpu.get("y").to(DEVICE) if g_cpu.get("y") is not None else None),
             "source": g_cpu.get("source","default")}
        sd,nl,_,_ = model.forward_full(g["x"], g["edges"])
        p = torch.sigmoid(nl).detach().cpu()
        y = g_cpu.get("y")
        if y is None or y.numel()!=p.numel(): 
            continue
        src = g_cpu.get("source","default")
        buf = per_source.setdefault(src, {"p":[], "y":[]})
        buf["p"].append(p)
        buf["y"].append(torch.as_tensor(y).float().view(-1))
    out={}
    for src,buf in per_source.items():
        if not buf["p"]: continue
        P=torch.cat(buf["p"]); Y=torch.cat(buf["y"])
        best_thr, best_f = 0.5, 0.0
        for thr in [i/100 for i in range(5,96,5)]:
            m=f1_from_probs(P, Y, thr)
            f=m["f1"] if m else 0.0
            if f>best_f: best_thr, best_f = thr, f
        out[src]=best_thr
    if not out: out={"default": 0.25}  # fallback
    return out

def counterfactual_ablate(x:torch.Tensor, idx:List[int], mode:str="mean")->torch.Tensor:
    # Stronger, less OOD than zeroing
    x_cf = x.clone()
    if not idx: return x_cf
    cols_mean = x.mean(dim=0, keepdim=True)
    x_cf[idx] = cols_mean.expand(len(idx), -1)
    return x_cf

def compute_CFAM(model, g_cpu, paths: List[BeamPath]):
    if not paths: return None
    x = g_cpu["x"].to(DEVICE).detach().requires_grad_(True)
    with torch.enable_grad():
        _, nl, _, _ = model.forward_full(x, {r:e.to(DEVICE) for r,e in g_cpu["edges"].items()})
        s = torch.sigmoid(nl).mean()
        s.backward()
        gn = x.grad.detach().abs().sum(dim=1)
    causal=set(n for bp in paths for n in bp.nodes if 0<=n<x.size(0))
    if not causal: return None
    mask=torch.zeros(x.size(0), dtype=torch.bool, device=gn.device)
    mask[torch.tensor(sorted(list(causal)), device=gn.device)] = True
    num=gn[mask].sum().item(); den=gn.sum().item()+1e-9
    return num/den

@torch.no_grad()
def compute_CCS(model, g_cpu, paths: List[BeamPath]):
    if not paths: return None
    x = g_cpu["x"].to(DEVICE)
    _, nl, _, _ = model.forward_full(x, {r:e.to(DEVICE) for r,e in g_cpu["edges"].items()})
    p0 = torch.sigmoid(nl).mean().item()
    causal = sorted(set(n for bp in paths for n in bp.nodes if 0<=n<x.size(0)))
    if not causal: return None
    cf = counterfactual_ablate(x, causal, mode="mean")
    _, nl2, _, _ = model.forward_full(cf, {r:e.to(DEVICE) for r,e in g_cpu["edges"].items()})
    p1 = torch.sigmoid(nl2).mean().item()
    return (p0 - p1) ** 2

# -------------------- Eval loop --------------------
def avg(lst): return (sum(lst)/len(lst)) if lst else None

def eval_split(name:str, data_dir:str, model, base_rel, add_summary, ckg, thr_by_source:Dict[str,float],
               limit=None)->Dict:
    files = list_files(data_dir, ("*.json","*.pt"))
    if limit is not None: files = files[:limit]

    # motif-set for quick matching
    motifs = set(tuple(m["rels"]) for m in ckg.get("motifs_topk", []) if isinstance(m.get("rels"), list) and len(m["rels"])==3)

    n_used = 0
    beam_len = []
    inter_ratio = []
    motif_hit = []
    P_list, Y_list = defaultdict(list), defaultdict(list)
    CFAM_vals, CCS_vals = [], []

    pbar = tqdm(files, desc=f"[eval] {data_dir}", unit="graph")
    for fp in pbar:
        try:
            gobj = safe_load(fp)
            g_cpu = normalize_graph(gobj, base_rel, add_summary)
            if g_cpu is None or g_cpu["x"] is None or g_cpu["x"].numel()==0: 
                continue
            g = {"x": g_cpu["x"].to(DEVICE),
                 "edges": {r:e.to(DEVICE) for r,e in g_cpu["edges"].items()},
                 "source": g_cpu.get("source","default")}
            with torch.no_grad(), (torch.amp.autocast('cuda', enabled=(USE_AMP and DEVICE=='cuda')) if DEVICE=='cuda' else torch.autocast("cpu", enabled=False)):
                sd, nl, h, es = model.forward_full(g["x"], g["edges"])
                seeds = pick_seeds(sd.detach(), g["edges"], SEED_K)
                p  = torch.sigmoid(nl)
                beams = run_beam_ckg_acc(p, es, g["edges"], seeds, ckg,
                                         width=BEAM_WIDTH, max_hops=BEAM_MAX_HOPS, alpha_node=ALPHA_NODE,
                                         lambda_edge=LAMBDA_EDGE, lambda_bigram=LAMBDA_BIGRAM, lambda_motif=LAMBDA_MOTIF)

            if not beams:
                continue
            top = beams[0]
            n_used += 1

            # beam stats
            beam_len.append(max(0, len(top.nodes)-1))
            # inter-proc ratio on the top beam:
            inter_hops = sum(1 for r in top.rels if r in {"CALL","ARG2PARAM","RET2CALL","RET2LHS"})
            inter_ratio.append(inter_hops / max(1,len(top.rels)))
            # motif hit?
            hit=False
            for i in range(2, len(top.rels)):
                tri = (top.rels[i-2], top.rels[i-1], top.rels[i])
                if tri in motifs:
                    hit=True; break
            motif_hit.append(1.0 if hit else 0.0)

            # detection metrics (only if labels exist)
            y = g_cpu.get("y")
            if y is not None and y.numel()==p.numel():
                src = g_cpu.get("source","default")
                P_list[src].append(p.detach().cpu())
                Y_list[src].append(torch.as_tensor(y).float().view(-1))

            # compute CFAM/CCS only for confident graphs (above threshold and path length)
            src_thr = thr_by_source.get(g_cpu.get("source","default"), thr_by_source.get("default", 0.25))
            mean_conf = p.detach().mean().item()
            if len(top.nodes)-1 >= CONF_GRAPH_MINLEN and mean_conf >= src_thr:
                cfam = compute_CFAM(model, g_cpu, [top])
                ccs  = compute_CCS(model, g_cpu,  [top])
                if cfam is not None: CFAM_vals.append(cfam)
                if ccs  is not None: CCS_vals.append(ccs)

        except Exception as ex:
            pbar.set_postfix_str(f"skip: {Path(fp).name}")

    # aggregate metrics
    overall = {"precision": None, "recall": None, "f1": None}
    # combine per-source P/R/F1 using calibrated thresholds
    precs, recs, f1s = [], [], []
    for src in P_list:
        P = torch.cat(P_list[src]); Y = torch.cat(Y_list[src])
        thr = thr_by_source.get(src, thr_by_source.get("default", 0.25))
        m = f1_from_probs(P, Y, thr)
        if m:
            precs.append(m["precision"]); recs.append(m["recall"]); f1s.append(m["f1"])
    if f1s:
        overall = {"precision": avg(precs), "recall": avg(recs), "f1": avg(f1s)}

    rep = {
        "split": name,
        "graphs_used": n_used,
        "beam": {
            "avg_len_edges": avg(beam_len),
            "interproc_ratio": avg(inter_ratio),
            "motif_hit_ratio": avg(motif_hit),
        },
        "overall": overall,
        "causal": {
            "CFAM_mean": avg(CFAM_vals),
            "CCS_mean":  avg(CCS_vals)
        }
    }
    return rep

# -------------------- Reports --------------------
def save_json(obj, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(obj, indent=2), encoding="utf-8")

def save_md_report(out_dir, valid_rep, test_rep, thr_by_source, ckg_used):
    lines=[]
    lines.append(f"# Evaluation Report\n")
    lines.append(f"- **Model:** `{Path(CKPT_PATH).resolve()}`")
    lines.append(f"- **CKG:** `{Path(CKG_PATH).resolve()}`")
    lines.append(f"- **Created at:** {time.strftime('%Y-%m-%d %H:%M:%S')}\n")

    def sec(rep):
        lines.append(f"## Split: {rep['split']}")
        lines.append(f"- Graphs used: **{rep['graphs_used']}**")
        b=rep["beam"]
        lines.append(f"- Beam avg length (edges): **{b['avg_len_edges']:.3f}**" if b['avg_len_edges'] is not None else "- Beam avg length (edges): n/a")
        lines.append(f"- Inter-procedural ratio: **{b['interproc_ratio']:.3f}**" if b['interproc_ratio'] is not None else "- Inter-procedural ratio: n/a")
        lines.append(f"- Motif-hit ratio: **{b['motif_hit_ratio']:.3f}**" if b['motif_hit_ratio'] is not None else "- Motif-hit ratio: n/a")
        o=rep["overall"]
        if o["f1"] is not None:
            lines.append(f"- Precision / Recall / F1: **{o['precision']:.3f} / {o['recall']:.3f} / {o['f1']:.3f}**")
        else:
            lines.append(f"- Precision / Recall / F1: n/a (no labels found)")
        c=rep["causal"]
        lines.append(f"- CFAM mean: **{c['CFAM_mean']:.6f}**" if c['CFAM_mean'] is not None else "- CFAM mean: n/a")
        lines.append(f"- CCS mean: **{c['CCS_mean']:.6f}**" if c['CCS_mean'] is not None else "- CCS mean: n/a")
        lines.append("")
    if valid_rep: sec(valid_rep)
    if test_rep:  sec(test_rep)

    lines.append("## Calibrated thresholds (by source)\n")
    for k,v in thr_by_source.items():
        lines.append(f"- {k}: **{v:.2f}**")
    lines.append("")

    # brief CKG summary used
    lines.append("## CKG summary used\n")
    rels = ckg_used.get("relations", [])
    ec   = ckg_used.get("edge_prior_count", {})
    lines.append(f"- Relations: {', '.join(rels)}")
    top = sorted(ec.items(), key=lambda kv: kv[1], reverse=True)[:10]
    if top:
        lines.append("- Top edge counts:")
        for r,c in top:
            lines.append(f"  - {r}: {c}")
    Path(out_dir, "beam_report.md").write_text("\n".join(lines), encoding="utf-8")

# -------------------- Main --------------------
def main():
    Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

    # 1) Load model & CKG
    assert os.path.exists(CKPT_PATH), f"Missing CKPT_PATH: {CKPT_PATH}"
    assert os.path.exists(CKG_PATH),  f"Missing CKG_PATH: {CKG_PATH}"
    ckg = json.loads(Path(CKG_PATH).read_text(encoding="utf-8"))
    model, base_rel, add_summary = load_model(CKPT_PATH)

    # 2) Preload VALID graphs for calibration (with labels if any)
    valid_graphs = list(iter_graphs(VALID_DIR, base_rel, add_summary, limit=EVAL_LIMIT_VALID))
    # 3) Calibrate per-source thresholds on valid
    thr_by_source = calibrate_thresholds(model, valid_graphs, base_rel, add_summary, limit=None)
    save_json(thr_by_source, Path(OUT_DIR,"thresholds.json"))

    # 4) Evaluate splits
    valid_rep = eval_split("valid", VALID_DIR, model, base_rel, add_summary, ckg, thr_by_source, limit=EVAL_LIMIT_VALID)
    test_rep  = eval_split("test",  TEST_DIR,  model, base_rel, add_summary, ckg, thr_by_source, limit=EVAL_LIMIT_TEST)

    # 5) Save reports
    reports = {"created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
               "ckpt": str(Path(CKPT_PATH).resolve()),
               "ckg":  str(Path(CKG_PATH).resolve()),
               "thresholds_by_source": thr_by_source,
               "valid": valid_rep, "test": test_rep}
    save_json(reports, Path(OUT_DIR,"reports.json"))
    save_json(ckg, Path(OUT_DIR,"ckg_used.json"))
    save_md_report(OUT_DIR, valid_rep, test_rep, thr_by_source, ckg)

    print(f"[SAVED] {OUT_DIR}")
    print("  - thresholds.json")
    print("  - reports.json")
    print("  - beam_report.md")
    print("  - ckg_used.json")

if __name__ == "__main__":
    main()


C:\Users\MSHUVO23\AppData\Local\Temp\ipykernel_29244\882722400.py:313: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location="cpu")


[env] torch=2.4.1+cu121 device=cuda


[load] Dataset/valid/hetero_ready_gcbert:   0%|          | 0/2906 [00:00<?, ?graph/s]C:\Users\MSHUVO23\AppData\Local\Temp\ipykernel_29244\882722400.py:59: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to t

[SAVED] out/eval_full
  - thresholds.json
  - reports.json
  - beam_report.md
  - ckg_used.json


TEST 4

In [6]:
# ================== Full Eval: conventional metrics + inter-proc metrics + CFAM/CCS ==================
# Config you confirmed:
CKPT_PATH   = r"out/cvul/1760989836/model.pt"     # your trained model
VALID_FEAT  = r"Dataset/valid/hetero_ready_gcbert"
TEST_FEAT   = r"Dataset/test/hetero_ready_gcbert"
VALID_LABEL = r"Dataset/valid/unified_aug"        # label JSONs
TEST_LABEL  = r"Dataset/test/unified_aug"
CKG_PATH    = r"ckg/ckg.json"                     # mined prior (optional, recommended)

# ---------------- Do not edit below unless you want to tweak behavior ----------------
import os, json, math, time, csv
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

# Eval params
SEED_K        = 8
BEAM_WIDTH    = 24
BEAM_MAX_HOPS = 5
ALPHA_NODE    = 0.7
LAMBDA_PRIOR  = 0.15
CALIB_MAX     = 400             # graphs for threshold calibration
PREFER_AUG_GRAPH = True         # use augmented JSON as graph if it **truly** contains features
ALLOW_STRUCT_FEATS_FALLBACK = True  # build structural features if none found

RELATIONS = ["DFG","CFG","CALL","ARG2PARAM","RET2CALL","RET2LHS"]
ADD_SUMMARY_EDGES = True        # adds DFG_THIN = DFG ∪ {ARG2PARAM,RET2CALL,RET2LHS}

OUT_DIR = Path(f"out/eval_{int(time.time())}").resolve()

# ---------------- Device ----------------
def pick_device(min_free_mb=256):
    if not torch.cuda.is_available(): return "cpu"
    try:
        free,_ = torch.cuda.mem_get_info()
        return "cuda" if (free//(1024**2)) >= min_free_mb else "cpu"
    except:
        return "cuda"
DEVICE = pick_device()
print(f"[env] torch={torch.__version__} device={DEVICE}")

# ---------------- IO ----------------
def safe_load(p: Path):
    p = Path(p)
    if p.suffix.lower()==".json":
        return json.loads(p.read_text(encoding="utf-8"))
    return torch.load(p, map_location="cpu")

def list_files(dirpath, exts=(".json",".pt")):
    d=Path(dirpath); out=[]
    for e in exts: out += sorted(d.glob(f"*{e}"))
    return out

def stems_in(dirpath):
    return {p.stem for p in list_files(dirpath)}

# ---------------- Graph normalization (robust) ----------------
def _to2d(x):
    t=torch.as_tensor(x).float()
    if t.ndim==1: t=t.view(-1,1)
    return t

def to_long_2(e):
    if e is None: return torch.zeros((2,0),dtype=torch.long)
    t=torch.as_tensor(e)
    if t.ndim==2 and t.shape[0]==2: return t.long().contiguous()
    if t.ndim==2 and t.shape[1]==2: return t.t().long().contiguous()
    if isinstance(e,(list,tuple)) and len(e)==2:
        s=torch.as_tensor(e[0]).view(-1).long()
        d=torch.as_tensor(e[1]).view(-1).long()
        return torch.stack([s,d],dim=0)
    if t.numel()==0: return torch.zeros((2,0),dtype=torch.long)
    raise RuntimeError("edge_index must be [2,E], [E,2], or (src,dst)")

def sanitize_edges(N, ei):
    if ei is None or ei.numel()==0: return torch.zeros((2,0),dtype=torch.long)
    s,d=ei
    m=(s>=0)&(s<N)&(d>=0)&(d<N)
    if m.any(): return torch.stack([s[m],d[m]],dim=0)
    return torch.zeros((2,0),dtype=torch.long)

# ----- feature extraction for HeteroData -----
def _first_2d_float(arr):
    t=torch.as_tensor(arr).float()
    if t.ndim==1: t=t.view(-1,1)
    return t

def _get_store_feat(store):
    # Try GraphCodeBERT fields first, then general names
    cand = ["gcbert","gcb_x","x","x_text","x_num","features","feat","emb"]
    for nm in cand:
        if hasattr(store, nm):
            val = getattr(store, nm)
            if val is not None:
                t=_first_2d_float(val)
                if t.numel()>0: return t
    if hasattr(store,"__dict__"):
        for nm in cand:
            if nm in store.__dict__ and store.__dict__[nm] is not None:
                t=_first_2d_float(store.__dict__[nm])
                if t.numel()>0: return t
    return None

def normalize_hetero(obj, relations, add_summary=True):
    node_stores=getattr(obj,"node_stores",None)
    edge_stores=getattr(obj,"edge_stores",None)
    if node_stores is None or edge_stores is None: return None

    candidates=[]
    for st in node_stores:
        key=getattr(st,"_key",None) or getattr(st,"type",None) or "code"
        feat=_get_store_feat(st)
        if feat is not None and feat.numel()>0:
            candidates.append((key, feat))
    if not candidates: return None

    nt,x=max(candidates, key=lambda kv: kv[1].size(0))
    N=x.size(0)

    E={r:torch.zeros((2,0),dtype=torch.long) for r in relations}
    for es in edge_stores:
        ei=getattr(es,"edge_index",None)
        if ei is None: continue
        src_t=getattr(es,"src_type",None)
        dst_t=getattr(es,"dst_type",None)
        if (src_t is not None and dst_t is not None) and not (src_t==nt and dst_t==nt):
            continue
        rel=getattr(es,"edge_type",None) or getattr(es,"_key",None)
        if isinstance(rel,(tuple,list)) and len(rel)==3:
            rel=str(rel[1])
        rel=str(rel).upper().split("__")[-1] if rel is not None else None
        if rel in E:
            E[rel]=sanitize_edges(N, to_long_2(ei))

    if add_summary:
        base=E.get("DFG", torch.zeros((2,0),dtype=torch.long))
        for r in ("ARG2PARAM","RET2CALL","RET2LHS"):
            if r in E and E[r].numel()>0:
                base=torch.cat([base,E[r]], dim=1)
        E["DFG_THIN"]=base

    # Try to pull y from same store (optional)
    y=None
    for st in node_stores:
        st_key=getattr(st,"_key",None) or getattr(st,"type",None)
        if st_key==nt:
            for lab in ["y","labels","target","targets","is_sink"]:
                if hasattr(st, lab):
                    yy=getattr(st, lab)
                    if yy is not None:
                        yy=torch.as_tensor(yy).float().view(-1)
                        if yy.numel()==N:
                            y=yy; break
            break

    return {"x":x,"edges":E,"y":y}

# ----- dict graphs -----
def coerce_x_any(obj):
    for k in ["gcbert","gcb_x","x","features","node_features","feat","emb","x_text","x_num","x_dense","x_numeric"]:
        if isinstance(obj,dict) and k in obj and obj[k] is not None:
            return _to2d(obj[k])
    if isinstance(obj,dict) and "nodes" in obj and isinstance(obj["nodes"],list) and obj["nodes"]:
        rows=[]
        for nd in obj["nodes"]:
            if not isinstance(nd,dict): continue
            for k in ["gcbert","gcb_x","x","feat","emb","features"]:
                if k in nd and nd[k] is not None:
                    rows.append(_to2d(nd[k])[0:1]); break
        if rows:
            d=max(r.size(1) for r in rows)
            rows=[F.pad(r,(0,d-r.size(1))) for r in rows]
            return torch.cat(rows,dim=0)
    return None

def build_edges_any(obj, N, relations, add_summary=True):
    E={r:torch.zeros((2,0),dtype=torch.long) for r in relations}
    if isinstance(obj,dict) and "edges" in obj and isinstance(obj["edges"],dict):
        for r in relations:
            if r in obj["edges"]:
                E[r]=sanitize_edges(N, to_long_2(obj["edges"][r]))
        if add_summary:
            base=E.get("DFG", torch.zeros((2,0),dtype=torch.long))
            for r in ("ARG2PARAM","RET2CALL","RET2LHS"):
                if E[r].numel(): base=torch.cat([base,E[r]], dim=1)
            E["DFG_THIN"]=base
        return E
    for r in relations:
        for k in [r, f"{r}_edge_index", f"edge_index_{r}", f"{r.lower()}_edge_index", f"edge_index_{r.lower()}"]:
            if isinstance(obj,dict) and k in obj:
                E[r]=sanitize_edges(N, to_long_2(obj[k])); break
    if add_summary:
        base=E.get("DFG", torch.zeros((2,0),dtype=torch.long))
        for r in ("ARG2PARAM","RET2CALL","RET2LHS"):
            if E[r].numel(): base=torch.cat([base,E[r]], dim=1)
        E["DFG_THIN"]=base
    return E

def normalize_graph(obj, relations, add_summary=True):
    # dict
    if isinstance(obj,dict):
        x=coerce_x_any(obj)
        if x is not None and x.numel()>0:
            N=x.size(0)
            E=build_edges_any(obj, N, relations, add_summary)
            y=None
            for k in ["y","labels","label","is_sink","target","targets"]:
                if k in obj and obj[k] is not None:
                    try:
                        yy=torch.as_tensor(obj[k]).float().view(-1)
                        if yy.numel()==N: y=yy
                    except: pass
                    break
            return {"x":x,"edges":E,"y":y}
    # PyG HeteroData or Data
    if "torch_geometric" in str(type(obj)):
        g_het=normalize_hetero(obj, relations, add_summary)
        if g_het is not None: return g_het
        if hasattr(obj,"x") and obj.x is not None:
            x=_to2d(obj.x); N=x.size(0)
            E={r:torch.zeros((2,0),dtype=torch.long) for r in relations}
            if hasattr(obj,"edge_index"): E["DFG"]=sanitize_edges(N, to_long_2(obj.edge_index))
            if add_summary: E["DFG_THIN"]=E["DFG"]
            return {"x":x,"edges":E,"y":None}
    return None

# ----- structural feature fallback (so evaluation never dies) -----
def structural_features(N:int, E:Dict[str,torch.Tensor], relations)->torch.Tensor:
    cols=[]
    for r in relations:
        ei=E.get(r); 
        if ei is None or ei.numel()==0:
            cols.append(torch.zeros(N,1)); cols.append(torch.zeros(N,1)); continue
        s,d=ei
        out=torch.zeros(N); out.index_add_(0, s, torch.ones_like(s, dtype=out.dtype))
        inn=torch.zeros(N); inn.index_add_(0, d, torch.ones_like(d, dtype=inn.dtype))
        cols.append(out.view(-1,1)); cols.append(inn.view(-1,1))
    deg=torch.zeros(N)
    for r in relations:
        ei=E.get(r)
        if ei is not None and ei.numel()>0:
            s,_=ei; deg.index_add_(0, s, torch.ones_like(s, dtype=deg.dtype))
    cols.append(deg.view(-1,1))
    X=torch.cat(cols, dim=1).float()
    # normalize
    if X.std()>0: X=(X - X.mean(0))/ (X.std(0)+1e-6)
    return X

# ---------------- Labels from unified_aug ----------------
def extract_y_from_label_obj(label_obj, N:int)->Optional[torch.Tensor]:
    # direct vectors
    for k in ["y","labels","label","is_sink","target","targets"]:
        if k in label_obj and label_obj[k] is not None:
            try:
                yy=torch.as_tensor(label_obj[k]).float().view(-1)
                if yy.numel()==N: return yy
            except: pass
    # nodes[].label
    if "nodes" in label_obj and isinstance(label_obj["nodes"],list):
        vals=[]
        for nd in label_obj["nodes"]:
            if not isinstance(nd,dict): vals.append(0.0); continue
            v=None
            for k in ["y","label","is_sink","target"]:
                if k in nd and nd[k] is not None:
                    try: v=float(nd[k]); break
                    except: pass
            vals.append(0.0 if v is None else v)
        if len(vals)==N: return torch.tensor(vals, dtype=torch.float32)
    # derive from sinks/paths
    y=torch.zeros(N, dtype=torch.float32); made=False
    def mark(idx):
        nonlocal made
        if isinstance(idx,(list,tuple)):
            for v in idx:
                if isinstance(v,(list,tuple)): mark(v)
                else:
                    j=int(v); 
                    if 0<=j<N: y[j]=1.0; made=True
        else:
            j=int(idx)
            if 0<=j<N: y[j]=1.0; made=True
    for k in ["sinks","sink_nodes","paths_idx","vulnerable_paths"]:
        if k in label_obj and label_obj[k] is not None:
            mark(label_obj[k])
    return y if made else None

def find_pair(stem:str, feat_dir:Path, label_dir:Path)->Tuple[Optional[Path], Optional[Path]]:
    feat=None
    for ext in (".json",".pt"):
        p=feat_dir/f"{stem}{ext}"
        if p.exists(): feat=p; break
    lab=None
    p=label_dir/f"{stem}.json"
    if p.exists(): lab=p
    else:
        cand=list(label_dir.glob(f"{stem}*.json"))
        if cand: lab=cand[0]
    return feat, lab

def choose_graph_source(stem:str, feat_dir:Path, label_dir:Path, prefer_aug=True)->Tuple[Optional[Path], Optional[Path], str]:
    feat, lab = find_pair(stem, feat_dir, label_dir)
    # prefer augmented JSON only if it actually contains features
    if prefer_aug and lab is not None:
        try:
            obj=safe_load(lab)
            g=normalize_graph(obj, RELATIONS, ADD_SUMMARY_EDGES)
            if g is not None and g["x"] is not None and g["x"].numel()>0:
                return lab, lab, "aug_graph"
        except: pass
    # else: use feature graph from hetero_ready_gcbert
    return feat, lab, "feat_graph"

def attach_labels_and_fix_features(graph_path:Path, label_path:Optional[Path])->Tuple[Optional[Dict], str]:
    """Load graph, attach labels; if no features, try structural fallback."""
    try:
        obj=safe_load(graph_path)
        g=normalize_graph(obj, RELATIONS, ADD_SUMMARY_EDGES)
        if g is None:
            return None, "bad_graph"
        # attach labels
        if label_path is not None and (g.get("y") is None or g["y"].numel()!=g["x"].size(0)):
            try:
                labobj=safe_load(label_path)
                y=extract_y_from_label_obj(labobj, g["x"].size(0))
                g["y"]=y
                lab_status = "label_ok" if y is not None else "label_missing_or_mismatch"
            except:
                lab_status = "label_load_error"
        else:
            lab_status = "y_in_graph" if g.get("y") is not None else "no_label_file"
        # feature fallback
        if (g["x"] is None or g["x"].numel()==0) and ALLOW_STRUCT_FEATS_FALLBACK:
            X=structural_features(g["edges"].get("DFG", torch.zeros((2,0))).max().item()+1 if g["edges"].get("DFG") is not None and g["edges"]["DFG"].numel()>0 else 0,
                                  g["edges"], RELATIONS)
            if X is not None and X.numel()>0:
                g["x"]=X
                return g, lab_status + "+struct_feats"
            return None, "no_features"
        return g, lab_status
    except:
        return None, "graph_load_error"

# ---------------- Model ----------------
class GraphBlock(nn.Module):
    def __init__(self, hidden, relations):
        super().__init__()
        self.relations=relations
        self.lin_rel=nn.ModuleDict({r:nn.Linear(hidden,hidden,bias=False) for r in relations})
        self.lin_self=nn.Linear(hidden,hidden)
    def forward(self,h,E):
        H=h
        for r in self.relations:
            ei=E.get(r)
            if ei is None or ei.numel()==0: continue
            s,d=ei
            msg=self.lin_rel[r](H)
            agg=torch.zeros_like(H)
            agg.index_add_(0, d, msg[s])
            H=H+agg
        return self.lin_self(H)

class CausalVulNet(nn.Module):
    def __init__(self, hidden=64, layers=3, relations=None, add_summary=True):
        super().__init__()
        rels=list(relations or RELATIONS)
        if add_summary and "DFG_THIN" not in rels: rels += ["DFG_THIN"]
        self.relations=rels
        self.proj_cache=nn.ModuleDict()
        self.blocks=nn.ModuleList([GraphBlock(hidden, self.relations) for _ in range(layers)])
        self.node_head=nn.Linear(hidden,1)
        self.seed_head=nn.Linear(hidden,1)
        self.rel_gate =nn.ParameterDict({r:nn.Parameter(torch.tensor(0.0)) for r in self.relations})
        self.edge_bilin=nn.Parameter(torch.empty(hidden,hidden)); nn.init.xavier_uniform_(self.edge_bilin)
        self.hidden=hidden; self.layers=layers
    def _proj(self,D):
        k=str(D)
        if k not in self.proj_cache:
            self.proj_cache[k]=nn.Linear(D,self.hidden).to(next(self.parameters()).device)
        return self.proj_cache[k]
    def encode(self,x,E):
        h=F.relu(self._proj(x.size(1))(x))
        for blk in self.blocks: h=F.elu(blk(h,E))
        return h
    def edge_scores(self,h,E):
        out={}
        for r,ei in E.items():
            if ei is None or ei.numel()==0:
                out[r]=torch.zeros((0,),device=h.device); continue
            s,d=ei
            out[r]=(h[s]@self.edge_bilin*h[d]).sum(dim=1) + self.rel_gate[r]
        return out
    def forward_full(self,x,E):
        seed_h=F.relu(self._proj(x.size(1))(x))
        seed_logit=self.seed_head(seed_h).squeeze(-1)
        h=self.encode(x,E)
        node_logit=self.node_head(h).squeeze(-1)
        edge_sc=self.edge_scores(h,E)
        return seed_logit,node_logit,h,edge_sc

def load_model(ckpt_path:str):
    ckpt=torch.load(ckpt_path, map_location="cpu")
    hidden = ckpt.get("hidden",64)
    layers = ckpt.get("layers",3)
    relations = ckpt.get("relations", RELATIONS+(["DFG_THIN"] if ADD_SUMMARY_EDGES else []))
    add_summary=("DFG_THIN" in relations)
    base_rel=[r for r in relations if r!="DFG_THIN"]
    model=CausalVulNet(hidden=hidden, layers=layers, relations=base_rel, add_summary=add_summary).to(DEVICE)
    model.load_state_dict(ckpt["state_dict"], strict=False)
    model.eval()
    return model, base_rel, add_summary

# ---------------- Beam + CKG ----------------
@dataclass
class BeamPath:
    score: float
    nodes: List[int]
    rels:  List[str]

def build_adj(E):
    adj_out={r:{} for r in E}; adj_in={r:{} for r in E}
    for r,ei in E.items():
        if ei is None or ei.numel()==0: continue
        s,d=ei; ss,dd=s.tolist(), d.tolist()
        for u,v in zip(ss,dd):
            adj_out[r].setdefault(u,[]).append(v)
            adj_in [r].setdefault(v,[]).append(u)
    return adj_out, adj_in

def pick_seeds(seed_logit, E, k):
    N=seed_logit.numel()
    deg=torch.zeros(N,device=seed_logit.device)
    for ei in E.values():
        if ei is None or ei.numel()==0: continue
        s,_=ei; deg.index_add_(0, s, torch.ones_like(s, dtype=deg.dtype))
    cand=torch.where(deg>0)[0]
    if cand.numel()==0: return torch.topk(seed_logit, k=min(k,N)).indices.tolist()
    k=min(k, cand.numel()); vals=seed_logit[cand]
    return cand[torch.topk(vals,k=k).indices].tolist()

def _edge_uv_scores(edge_sc,E,N):
    uv={}
    for r,ei in E.items():
        if ei is None or ei.numel()==0: uv[r]={}; continue
        s,d=ei; es=edge_sc[r].detach().float()
        mp={}
        for i in range(s.numel()):
            u=int(s[i]); v=int(d[i])
            if 0<=u<N and 0<=v<N:
                val=float(es[i].item())
                mp[(u,v)] = max(mp.get((u,v), val), val)
        uv[r]=mp
    return uv

def run_beam(p, edge_sc, E, seeds, width=24, max_hops=5, alpha_node=0.7, ckg=None, lambda_prior=0.15):
    N=p.numel()
    adj_out, adj_in = build_adj(E)
    uv=_edge_uv_scores(edge_sc,E,N)
    def clog(x): return float(torch.log(x.clamp(1e-9,1-1e-9)))
    if ckg is not None:
        ep = ckg.get("edge_prior_prob", {})
        bp = ckg.get("bigram_prob", {})
        def lp_edge(r):      return math.log(max(float(ep.get(r,1e-6)), 1e-9))
        def lp_bigram(a,b):  return math.log(max(float(bp.get(a, {}).get(b,1e-6)), 1e-9))
    beams=[BeamPath(clog(p[s]), [int(s)], []) for s in seeds if 0<=int(s)<N]
    if not beams: return []
    out=[]
    for _ in range(max_hops):
        nxt=[]
        for b in beams:
            u=b.nodes[-1]
            cand=[]
            for r in E.keys():
                for v in adj_out[r].get(u, []): cand.append((r,u,v))
                for v in adj_in [r].get(u, []): cand.append((r,v,u))
            if not cand: out.append(b); continue
            prev_r=b.rels[-1] if b.rels else None
            for (r,uu,vv) in cand:
                if not (0<=vv<N): continue
                es=uv.get(r,{}).get((uu,vv), 0.0)
                sc=b.score + alpha_node*clog(p[vv]) + (1-alpha_node)*es
                if ckg is not None:
                    sc += lambda_prior*lp_edge(r)
                    if prev_r is not None:
                        sc += lambda_prior*lp_bigram(prev_r, r)
                nxt.append(BeamPath(sc, b.nodes+[vv], b.rels+[r]))
        if not nxt: break
        nxt.sort(key=lambda z:z.score, reverse=True)
        beams=nxt[:width]
    out.extend(beams); out.sort(key=lambda z:z.score, reverse=True)
    return out[:width]

# ---------------- Metrics ----------------
def f1_from_probs(p:torch.Tensor, y:torch.Tensor, thr:float):
    yb=(y>0.5); pb=(p>thr)
    tp=(pb&yb).sum().item(); fp=(pb&~yb).sum().item(); fn=(~pb&yb).sum().item()
    prec=tp/(tp+fp+1e-9); rec=tp/(tp+fn+1e-9); f1=2*prec*rec/(prec+rec+1e-9)
    return {"precision":prec,"recall":rec,"f1":f1}

def interproc_mask(E, N:int):
    inter={"CALL","ARG2PARAM","RET2CALL","RET2LHS"}
    m=torch.zeros(N, dtype=torch.bool)
    for r in inter:
        ei=E.get(r)
        if ei is None or ei.numel()==0: continue
        s,d=ei
        m[s]=True; m[d]=True
    return m

def compute_cfam_slice(model, g_cpu, paths:List[BeamPath]):
    if not paths: return None
    x=g_cpu["x"].to(DEVICE).detach().requires_grad_(True)
    idx=sorted({n for bp in paths for n in bp.nodes if 0<=n<x.size(0)})
    if not idx: return None
    with torch.enable_grad():
        _,nl,_,_=model.forward_full(x, {r:e.to(DEVICE) for r,e in g_cpu["edges"].items()})
        s=torch.sigmoid(nl).mean(); s.backward()
        gn=x.grad.detach().abs().sum(dim=1)
    mask=torch.zeros(x.size(0), dtype=torch.bool, device=gn.device)
    mask[torch.tensor(idx,device=gn.device)]=True
    return (gn[mask].sum().item())/(gn.sum().item()+1e-9)

def compute_ccs_slice(model, g_cpu, paths:List[BeamPath]):
    if not paths: return None
    x=g_cpu["x"].to(DEVICE)
    idx=sorted({n for bp in paths for n in bp.nodes if 0<=n<x.size(0)})
    if not idx: return None
    with torch.no_grad():
        _,nl,_,_=model.forward_full(x, {r:e.to(DEVICE) for r,e in g_cpu["edges"].items()})
        p0=torch.sigmoid(nl)[idx].mean().item()
    cf=x.clone()
    cf[torch.tensor(idx,device=cf.device)] = 0.0
    with torch.no_grad():
        _,nl2,_,_=model.forward_full(cf, {r:e.to(DEVICE) for r,e in g_cpu["edges"].items()})
        p1=torch.sigmoid(nl2)[idx].mean().item()
    return (p0-p1)**2

def interproc_ratio(paths:List[BeamPath])->float:
    if not paths: return 0.0
    inter={"CALL","ARG2PARAM","RET2CALL","RET2LHS"}
    hops=0; inter_hops=0
    for bp in paths:
        for r in bp.rels:
            hops+=1
            if r in inter: inter_hops+=1
    return inter_hops/max(1,hops)

# ---------------- Calibration ----------------
def calibrate_thresholds(model, pairs, relations, add_summary, max_items=400):
    bufP=[]; bufY=[]
    pbar=tqdm(pairs[:max_items], desc="[calibrate]", unit="graph")
    for graph_path, label_path, src in pbar:
        try:
            g, status = attach_labels_and_fix_features(graph_path, label_path)
            if g is None or g.get("y") is None or g["y"].numel()!=g["x"].size(0): continue
            dev={"x":g["x"].to(DEVICE), "edges":{r:e.to(DEVICE) for r,e in g["edges"].items()}}
            with torch.no_grad():
                _,nl,_,_=model.forward_full(dev["x"], dev["edges"])
                p=torch.sigmoid(nl).detach().cpu().view(-1)
            bufP.append(p); bufY.append(g["y"].float().cpu().view(-1))
        except:
            pass
    if not bufP:
        return 0.5
    P=torch.cat(bufP); Y=torch.cat(bufY)
    best_thr, best_f=0.5, -1
    for thr in [i/100 for i in range(5,96,5)]:
        m=f1_from_probs(P,Y,thr)
        if m["f1"]>best_f: best_thr, best_f = thr, m["f1"]
    return best_thr

# ---------------- Pairing ----------------
def enumerate_pairs(feat_dir:str, label_dir:str, prefer_aug=True):
    feat_dir=Path(feat_dir); label_dir=Path(label_dir)
    stems = sorted(stems_in(feat_dir) | stems_in(label_dir))
    pairs=[]
    for s in stems:
        graph_path, label_path, src = choose_graph_source(s, feat_dir, label_dir, prefer_aug)
        if graph_path is None and label_path is None:
            continue
        pairs.append((graph_path or label_path, label_path, src))
    return pairs

def to_device_graph(g):
    return {"x":g["x"].to(DEVICE),
            "edges":{r:e.to(DEVICE) for r,e in g["edges"].items()},
            "y": (g["y"].to(DEVICE) if g.get("y") is not None else None)}

# ---------------- Eval split ----------------
def eval_split(split_name, feat_dir, label_dir, model, relations, add_summary, ckg):
    pairs = enumerate_pairs(feat_dir, label_dir, prefer_aug=PREFER_AUG_GRAPH)
    if not pairs:
        return {"n_graphs":0, "n_graphs_used_for_metrics":0}

    thr=calibrate_thresholds(model, pairs, relations, add_summary, max_items=min(CALIB_MAX,len(pairs)))

    stats_all={"precision":0.0,"recall":0.0,"f1":0.0}
    stats_ip ={"precision":0.0,"recall":0.0,"f1":0.0}
    cnt_all=0; cnt_ip=0
    inter_list=[]; blen=[]; cfam_vals=[]; ccs_vals=[]
    dbg_counts=Counter()
    debug_rows=[["stem","graph_path","label_path","src","N","label_status","pos_count","used_for_metrics","used_for_ip_metrics"]]

    pbar=tqdm(pairs, desc=f"[eval:{split_name}]", unit="graph")
    for graph_path, label_path, src in pbar:
        stem=Path(graph_path).stem if graph_path else (Path(label_path).stem if label_path else "<none>")
        g, lab_status = attach_labels_and_fix_features(graph_path, label_path)
        if g is None:
            dbg_counts[lab_status]+=1
            debug_rows.append([stem, str(graph_path), str(label_path), src, None, lab_status, None, 0, 0])
            continue

        dev=to_device_graph(g)
        try:
            with torch.no_grad():
                sd,nl,h,es=model.forward_full(dev["x"], dev["edges"])
                p=torch.sigmoid(nl)
                seeds=pick_seeds(sd, dev["edges"], SEED_K)
                beams=run_beam(p, es, dev["edges"], seeds,
                               width=BEAM_WIDTH, max_hops=BEAM_MAX_HOPS,
                               alpha_node=ALPHA_NODE, ckg=ckg, lambda_prior=LAMBDA_PRIOR)
        except Exception as ex:
            dbg_counts[f"model_err:{type(ex).__name__}"]+=1
            debug_rows.append([stem, str(graph_path), str(label_path), src, int(g['x'].size(0)), "model_error", None, 0, 0])
            continue

        # beam stats + CFAM/CCS
        if beams:
            blen.append(sum(len(bp.nodes)-1 for bp in beams)/len(beams))
            inter_list.append(interproc_ratio(beams))
            cf=compute_cfam_slice(model, g, beams); cc=compute_ccs_slice(model, g, beams)
            if cf is not None: cfam_vals.append(cf)
            if cc is not None: ccs_vals.append(cc)

        # conventional metrics (all nodes)
        used_all=0; used_ip=0
        if g.get("y") is not None and g["y"].numel()==p.numel():
            m=f1_from_probs(p.detach().cpu(), g["y"].cpu().float(), thr)
            stats_all["precision"]+=m["precision"]; stats_all["recall"]+=m["recall"]; stats_all["f1"]+=m["f1"]
            cnt_all+=1; used_all=1

            # inter-proc restricted metrics
            ipm=interproc_mask(g["edges"], g["x"].size(0))
            if ipm.any():
                m_ip=f1_from_probs(p[ipm].detach().cpu(), g["y"][ipm].cpu().float(), thr)
                stats_ip["precision"]+=m_ip["precision"]; stats_ip["recall"]+=m_ip["recall"]; stats_ip["f1"]+=m_ip["f1"]
                cnt_ip+=1; used_ip=1

            dbg_counts[lab_status]+=1
        else:
            dbg_counts[lab_status]+=1

        debug_rows.append([stem, str(graph_path), str(label_path), src, int(g["x"].size(0)), lab_status,
                           (float((g["y"]>0.5).sum().item()) if g.get("y") is not None else None), used_all, used_ip])

    rep={
        "split":split_name,
        "n_graphs": len(pairs),
        "n_graphs_used_for_metrics": cnt_all,
        "n_graphs_used_for_ip_metrics": cnt_ip,
        "threshold_used": thr,
        "overall": {
            "precision": (stats_all["precision"]/cnt_all if cnt_all else None),
            "recall":    (stats_all["recall"]/cnt_all    if cnt_all else None),
            "f1":        (stats_all["f1"]/cnt_all        if cnt_all else None),
        },
        "interproc_only": {
            "precision": (stats_ip["precision"]/cnt_ip if cnt_ip else None),
            "recall":    (stats_ip["recall"]/cnt_ip    if cnt_ip else None),
            "f1":        (stats_ip["f1"]/cnt_ip        if cnt_ip else None),
        },
        "beam": {
            "avg_steps": (sum(blen)/len(blen) if blen else None),
            "interproc_ratio": (sum(inter_list)/len(inter_list) if inter_list else None)
        },
        "CFAM_slice_mean": (sum(cfam_vals)/len(cfam_vals) if cfam_vals else None),
        "CCS_slice_mean":  (sum(ccs_vals)/len(ccs_vals)   if ccs_vals else None),
        "debug_label_counts": dict(dbg_counts)
    }
    # Save debug CSV
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    with open(OUT_DIR/f"debug_label_attach_{split_name}.csv","w",newline="",encoding="utf-8") as f:
        writer=csv.writer(f); writer.writerows(debug_rows)
    return rep

# ---------------- CKG load & Reports ----------------
def load_ckg_or_none(path):
    if path is None or not os.path.exists(path): return None
    try:
        return json.loads(Path(path).read_text(encoding="utf-8"))
    except:
        return None

def save_reports(out_dir, reports, ckg_used):
    out_dir.mkdir(parents=True, exist_ok=True)
    Path(out_dir/"reports.json").write_text(json.dumps(reports, indent=2), encoding="utf-8")

    lines=[]
    lines.append("# Beam + ACC + CKG: Evaluation Summary\n")
    for key in ["valid","test"]:
        if key in reports:
            r=reports[key]; ov=r.get("overall",{}); ip=r.get("interproc_only",{})
            lines.append(f"## {key.title()}")
            lines.append(f"- Paired graphs: {r.get('n_graphs')}")
            lines.append(f"- Used (all-node metrics): {r.get('n_graphs_used_for_metrics')}")
            lines.append(f"- Used (inter-proc metrics): {r.get('n_graphs_used_for_ip_metrics')}")
            lines.append(f"- Threshold: {r.get('threshold_used')}")
            lines.append(f"- Overall P/R/F1: {ov.get('precision')} / {ov.get('recall')} / {ov.get('f1')}")
            lines.append(f"- Inter-proc P/R/F1: {ip.get('precision')} / {ip.get('recall')} / {ip.get('f1')}")
            bm=r.get("beam",{})
            lines.append(f"- Beam steps (avg): {bm.get('avg_steps')}")
            lines.append(f"- Inter-proc hop ratio: {bm.get('interproc_ratio')}")
            lines.append(f"- CFAM_slice mean: {r.get('CFAM_slice_mean')}")
            lines.append(f"- CCS_slice mean: {r.get('CCS_slice_mean')}")
            lines.append(f"- Label attach counts: {r.get('debug_label_counts')}\n")
    Path(out_dir/"beam_report.md").write_text("\n".join(lines), encoding="utf-8")

    if ckg_used is not None:
        with open(out_dir/"ckg_used.json","w",encoding="utf-8") as f:
            json.dump(ckg_used, f, indent=2)
        md = "# CKG Prior Used\n"
        md+= f"- Relations: {len(ckg_used.get('relations',[]))}\n"
        md+= f"- Motifs: {len(ckg_used.get('motifs_topk',[]))}\n"
        Path(out_dir/"ckg_used_report.md").write_text(md, encoding="utf-8")

# ---------------- MAIN ----------------
def main():
    assert os.path.exists(CKPT_PATH), f"Missing CKPT: {CKPT_PATH}"
    model, base_rel, add_summary = load_model(CKPT_PATH)
    ckg = load_ckg_or_none(CKG_PATH)

    reports={}
    if os.path.isdir(VALID_FEAT) and os.path.isdir(VALID_LABEL):
        reports["valid"]=eval_split("valid", VALID_FEAT, VALID_LABEL, model, base_rel, add_summary, ckg)
        print("[valid]", reports["valid"])
    else:
        print("[valid] skipped (dir missing)")

    if os.path.isdir(TEST_FEAT) and os.path.isdir(TEST_LABEL):
        reports["test"]=eval_split("test", TEST_FEAT, TEST_LABEL, model, base_rel, add_summary, ckg)
        print("[test]", reports["test"])
    else:
        print("[test] skipped (dir missing)")

    save_reports(OUT_DIR, reports, ckg)
    print(f"[done] all reports saved to: {OUT_DIR}")

if __name__=="__main__":
    main()


[env] torch=2.4.1+cu121 device=cuda


C:\Users\MSHUVO23\AppData\Local\Temp\ipykernel_29244\971239273.py:408: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt=torch.load(ckpt_path, map_location="cpu")
[calibrat

[valid] {'split': 'valid', 'n_graphs': 2906, 'n_graphs_used_for_metrics': 2905, 'n_graphs_used_for_ip_metrics': 2835, 'threshold_used': 0.05, 'overall': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0}, 'interproc_only': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0}, 'beam': {'avg_steps': 4.901927047487956, 'interproc_ratio': 0.41204117916952965}, 'CFAM_slice_mean': 0.03713512300016863, 'CCS_slice_mean': 2.103382822502565e-05, 'debug_label_counts': {'label_ok': 2905, 'no_label_file': 1}}


[eval:test]: 100%|██████████| 2915/2915 [3:18:43<00:00,  4.09s/graph]    

[test] {'split': 'test', 'n_graphs': 2915, 'n_graphs_used_for_metrics': 2915, 'n_graphs_used_for_ip_metrics': 2837, 'threshold_used': 0.05, 'overall': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0}, 'interproc_only': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0}, 'beam': {'avg_steps': 4.88336192109777, 'interproc_ratio': 0.4147284162378449}, 'CFAM_slice_mean': 0.03802854397785558, 'CCS_slice_mean': 2.992976884928558e-05, 'debug_label_counts': {'label_ok': 2915}}
[done] all reports saved to: C:\Users\MSHUVO23\Desktop\Thesis Research\Thesis-causal-vul\notebooks\out\eval_1761512346
